# Segmentasi Nasabah Bank (Customer Segmentation): End-to-End Clustering

**Kasus bisnis:** bank ingin mengelompokkan nasabah berdasarkan **perilaku dan nilai** mereka, sehingga produk, kanal, dan program bisa disesuaikan per segmen (bukan satu pendekatan untuk semua nasabah).

Berbeda dengan klasifikasi, **clustering tidak punya label jawaban**. Karena itu evaluasinya bertumpu pada:

| Pertanyaan | Metode di notebook ini |
|---|---|
| Apakah data memang punya struktur kelompok? | Hopkins statistic, uji permutasi vs data nol |
| Berapa jumlah segmen yang tepat? | Elbow, silhouette, Calinski-Harabasz, Davies-Bouldin, BIC/AIC + **stabilitas** + rentang bisnis |
| Model mana yang lebih baik: K-Means atau Gaussian Mixture? | Metrik internal, **bootstrap stability** (ARI & Jaccard per cluster), **split-half “cross-validation”**, uji permutasi |
| Apakah segmen berbeda secara bermakna? | Kruskal-Wallis + ε², chi-square + Cramér's V (koreksi Holm) |
| Apakah segmen tetap berlaku di periode berikutnya? | **Validasi out-of-time** (pengganti backtesting): PSI distribusi segmen, refit + pencocokan Hungarian, drift centroid, CSI fitur |
| Apakah segmen bisa ditindaklanjuti bisnis? | Profil, persona, rekomendasi produk, kontribusi dana & kredit per segmen |

Hasil akhirnya model disimpan sebagai **JSON**, lalu disajikan lewat **FastAPI** (`app/`) dan **Docker**.

> Setiap cell kode didahului kotak **🔄 Alur data** (Input → Proses → Output → Berikutnya).

## 0. Peta Alur Data

```
 data/customers.csv (12.000 nasabah × 18 kolom, snapshot Jan–Des 2025)
          │  pd.read_csv
          ▼
 df ─────────────────────────► EDA (Bagian 3): hanya dibaca
          │  split WAKTU (bukan acak)
          ▼
 dev: Jan–Sep 2025 (±9.000)                    OOT: Okt–Des 2025 (±3.000) ← dikunci sampai Bagian 13
     │
     │  preprocess: imputasi → log1p (7 kolom uang) → StandardScaler
     │              + one-hot kanal × bobot 0,5            12 kolom mentah ──► 15 kolom angka
     ▼
 X_dev ──► uji kecenderungan cluster (Bagian 6): Hopkins
     │
     ▼
 2 model × k = 2…10 (Bagian 7–8): metrik internal + stabilitas → k terbaik per model
     │
     ▼
 Model statistik kandidat (Bagian 8.1): VIF fitur + OLS fitur ~ segmen (intercept, koefisien, p-value)
     │
     ▼
 Evaluasi mendalam 2 kandidat (Bagian 9): silhouette, bootstrap, split-half, uji permutasi → model terbaik
     │
     ▼
 Model final + penamaan persona otomatis (Bagian 10) ──► profil & uji beda segmen (Bagian 11)
     │                                                 ──► validasi eksternal (Bagian 12, khusus data sintetis)
     ▼
 Validasi out-of-time di OOT (Bagian 13) ──► rekomendasi bisnis & ringkasan evaluasi (Bagian 14)
     │
     ▼
 model.json (Bagian 15) ──► validasi identik (Bagian 16) ──► API /segment (Bagian 17)
```

### Siklus hidup variabel utama

| Variabel | Bentuk | Dibuat | Dipakai | Isi |
|---|---|---|---|---|
| `df` | 12.000 × 18 | Bagian 2 | 3–4 | Data mentah |
| `dev`, `oot` | ±9.000 / ±3.000 baris | Bagian 4 | 5–16 | Periode pengembangan / out-of-time |
| `X_dev` | ±9.000 × 15 | Bagian 5 | 6–9 | Data dev setelah preprocessing |
| `scan` | 18 baris (2 model × 9 k) | Bagian 8 | 8 | Metrik per k |
| `candidates` | 2 pipeline | Bagian 9 | 9–13 | K-Means & GMM dengan k terbaik masing-masing |
| `final_model`, `SEGMENTS` | Pipeline, dict | Bagian 10 | 10–16 | Model terpilih & persona tiap segmen |
| `profile` | k × banyak kolom | Bagian 11 | 11, 14, 15 | Profil segmen |
| `oot_labels`, `oot_summary` | ±3.000, tabel | Bagian 13 | 13–14 | Hasil validasi out-of-time |
| `evaluation_summary` | tabel | Bagian 14 | 15 | Rangkuman semua bukti statistik |
| `model.json` | file | Bagian 15 | 16, API | Pipeline + persona + metadata evaluasi |

**Aturan yang dijaga:**
1. Kolom `_true_persona` (label asli dari generator) **tidak pernah** dipakai untuk melatih atau memilih model. Kolom ini hanya dipakai di Bagian 12 sebagai validasi eksternal, yang tidak mungkin dilakukan di data nyata.
2. Preprocessing dan model hanya di-*fit* di data **dev**. Data OOT hanya diberi label (`transform`/`predict`).
3. Pemilihan k dan model memakai aturan yang ditulis **sebelum** melihat hasil (Bagian 8 & 9).

## 1. Import & Konfigurasi

> **🔄 Alur data: Import & Konfigurasi**
>
> - **Input:** Belum ada data.
> - **Proses:**
>   1. Import library: pandas/numpy, matplotlib/seaborn, scipy (uji statistik), scikit-learn (K-Means, GMM, PCA, metrik).
>   2. Import modul proyek: `src/features.py` (daftar fitur, preprocessing, binning profil), `src/evaluation.py` (Hopkins, bootstrap, split-half, uji permutasi, Kruskal-Wallis, PSI, dll.), `src/personas.py` (penamaan segmen), `src/json_model.py` (ekspor/inferensi JSON).
>   3. Konstanta: `OOT_START = 2025-10`, `K_RANGE = 2…10`, `BUSINESS_K = (4, 8)`, `STABILITY_MIN = 0,75`, jumlah ulangan bootstrap/split-half/permutasi, sampel silhouette 5.000.
> - **Output:** Fungsi & konstanta siap dipakai.
> - **Berikutnya:** Baca CSV.

In [ ]:
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import statsmodels.api as sm
from scipy import stats
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_samples
from sklearn.mixture import GaussianMixture
from sklearn.pipeline import Pipeline
from statsmodels.stats.outliers_influence import variance_inflation_factor

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))

from src.evaluation import (  # noqa: E402
    bootstrap_stability,
    cramers_v,
    games_howell,
    holm_correction,
    hopkins_statistic,
    internal_metrics,
    kruskal_epsilon,
    match_clusters,
    permutation_test_silhouette,
    permute_columns,
    psi_categorical,
    psi_numeric,
    split_half_stability,
    stability_label,
    stability_verdict,
)
from src.features import (  # noqa: E402
    CAT_COLS,
    CAT_WEIGHT,
    DATE_COL,
    FEATURES,
    ID_COL,
    LINEAR_COLS,
    LOG_COLS,
    PROFILE_BINS,
    PROFILE_CAT_COLS,
    TRUE_LABEL_COL,
    add_profile_bins,
    build_preprocessor,
)
from src.json_model import JsonSegmentModel, export_pipeline, save_json  # noqa: E402
from src.personas import name_segments  # noqa: E402

RANDOM_STATE = 42
OOT_START = "2025-10"          # Okt–Des 2025 = out-of-time
K_RANGE = range(2, 11)         # k yang dicoba
BUSINESS_K = (4, 8)            # jumlah segmen yang masih bisa dikelola tim marketing
STABILITY_MIN = 0.75           # ambang stabil (Hennig, 2007)
N_BOOT_SCAN, N_BOOT_FINAL = 10, 50
N_SPLIT_HALF, N_PERM = 20, 99
SIL_SAMPLE = 5000              # silhouette O(n²): saat scan k memakai sampel 5.000 baris
NUMERIC_COLS = LOG_COLS + LINEAR_COLS

DATA_PATH = PROJECT_DIR / "data" / "customers.csv"
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250)
print(f"Python {platform.python_version()} | scikit-learn {sklearn.__version__} | pandas {pd.__version__}")

## 2. Load Data

> **🔄 Alur data: Load Data**
>
> - **Input:** `data/customers.csv` (dibuat oleh `scripts/generate_data.py`).
> - **Proses:**
>   1. Cek file ada, lalu `pd.read_csv`.
> - **Output:** `df` berukuran **12.000 × 18**. Satu baris = satu nasabah pada satu bulan snapshot.
> - **Berikutnya:** Cek skema terhadap kamus data.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"{DATA_PATH} belum ada. Jalankan dulu: python scripts/generate_data.py")
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

### Kamus Data

| Kolom | Tipe | Keterangan | Peran |
|---|---|---|---|
| `customer_id` | ID | ID nasabah | identitas (tidak dipakai model) |
| `snapshot_month` | tanggal | Bulan pengambilan data (2025-01 … 2025-12) | split waktu dev/OOT |
| `monthly_salary` | numerik | Gaji/penghasilan bulanan (juta Rp), ±4% kosong | **model** (log) + profil (binning) |
| `avg_balance` | numerik | Rata-rata saldo tabungan/giro (juta Rp) | **model** (log) |
| `investment_balance` | numerik | Saldo deposito/reksa dana/obligasi (juta Rp) | **model** (log) |
| `monthly_txn_count` | numerik | Jumlah transaksi per bulan | **model** (log) |
| `monthly_txn_amount` | numerik | Nominal transaksi per bulan (juta Rp) | **model** (log) |
| `credit_card_spend` | numerik | Belanja kartu kredit per bulan (juta Rp; 0 = tidak punya/tidak pakai) | **model** (log) |
| `loan_outstanding` | numerik | Sisa pinjaman (juta Rp) | **model** (log) |
| `age` | numerik | Umur | **model** + profil (binning) |
| `tenure_years` | numerik | Lama menjadi nasabah (tahun) | **model** |
| `num_products` | numerik | Jumlah produk bank yang dimiliki | **model** |
| `digital_txn_ratio` | numerik | Porsi transaksi lewat kanal digital (0–1) | **model** |
| `preferred_channel` | kategorikal | mobile / internet_banking / branch / atm | **model** (one-hot × bobot) |
| `occupation` | kategorikal | Pekerjaan (8 kategori seperti proyek klasifikasi, tanpa `unemployed` karena semua nasabah di data ini berpenghasilan) | profil |
| `employee_status` | kategorikal | permanent / contract / outsourcing / not_applicable | profil |
| `city_tier` | kategorikal | tier1 (kota besar) / tier2 / tier3 | profil |
| `_true_persona` | label | Persona asli dari generator | **hanya** validasi eksternal (Bagian 12) |

**Kenapa pekerjaan, status karyawan, dan kota tidak dipakai model?** Ini *behavioral segmentation*: segmen dibentuk dari **apa yang nasabah lakukan** (saldo, transaksi, kredit, kanal), lalu demografi dipakai untuk **menjelaskan** siapa mereka. Kalau 9 kategori pekerjaan ikut di-one-hot, jarak antar nasabah akan didominasi pekerjaan dan segmen hanya menjadi "kelompok pekerjaan".

> **🔄 Alur data: Validasi Skema**
>
> - **Input:** `df`.
> - **Proses:**
>   1. `assert` memastikan kolom `df` persis = 12 fitur model + 3 kolom profil + ID + bulan + label asli.
>   2. `assert` memastikan `customer_id` unik (1 baris = 1 nasabah).
>   3. `df.info()` menampilkan tipe dan jumlah nilai terisi.
> - **Output:** Skema valid, `customer_id` unik. Hanya `monthly_salary` yang tidak penuh (11.557 dari 12.000).
> - **Berikutnya:** Hitung kualitas data.

In [ ]:
assert set(FEATURES + PROFILE_CAT_COLS + [ID_COL, DATE_COL, TRUE_LABEL_COL]) == set(df.columns)
assert df[ID_COL].is_unique
df.info()

## 3. Exploratory Data Analysis

> **🔄 Alur data: Cek Kualitas Data**
>
> - **Input:** `df`.
> - **Proses:**
>   1. Per kolom: tipe, jumlah & % missing, jumlah nilai unik, **% nilai nol** (penting untuk clustering: banyak nol = distribusi menumpuk).
> - **Output:** Tabel `quality`. Missing hanya `monthly_salary` 3,7%. Nilai nol: `credit_card_spend` **47%**, `loan_outstanding` 15%, `investment_balance` 10%. Duplikat 0. `occupation` 8 kategori.
> - **Berikutnya:** Ukur kemiringan distribusi untuk memutuskan transformasi.

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(),
    "porsi_nol_%": (df.eq(0).mean() * 100).round(2),
})
print("Baris duplikat:", int(df.duplicated().sum()))
quality

> **🔄 Alur data: Skewness Sebelum & Sesudah log1p**
>
> - **Input:** `df[NUMERIC_COLS]` (11 kolom).
> - **Proses:**
>   1. Skewness mentah vs setelah `log1p` per kolom, plus median, mean, max.
>   2. Kolom diberi tanda apakah termasuk `LOG_COLS` (di-log di pipeline).
> - **Output:** Tabel `skew`. Skewness mentah 1,8–9,2 untuk 7 kolom uang (tertinggi `investment_balance` 9,2), turun ke 0,1–1,4 setelah log1p. 4 kolom linear sudah cukup simetris (|skew| ≤ 1,05).
> - **Berikutnya:** Visualisasikan efek log.

In [ ]:
skew = pd.DataFrame({
    "skewness_mentah": df[NUMERIC_COLS].skew(),
    "skewness_setelah_log1p": np.log1p(df[NUMERIC_COLS]).skew(),
    "median": df[NUMERIC_COLS].median(),
    "mean": df[NUMERIC_COLS].mean(),
    "max": df[NUMERIC_COLS].max(),
}).round(2)
skew["di-log"] = np.where(skew.index.isin(LOG_COLS), "ya", "tidak")
skew.sort_values("skewness_mentah", ascending=False)

> **🔄 Alur data: Histogram Mentah vs log1p**
>
> - **Input:** `df[LOG_COLS]` (7 kolom).
> - **Proses:**
>   1. Baris atas: histogram nilai mentah. Baris bawah: histogram setelah `log1p`.
> - **Output:** Grafik: distribusi mentah menumpuk di kiri dengan ekor panjang; setelah log jauh lebih simetris. Lonjakan di 0 (nasabah tanpa kartu kredit/investasi/pinjaman) tetap terlihat sebagai batang tersendiri.
> - **Berikutnya:** Lihat fitur linear & kategorikal.

In [ ]:
fig, axes = plt.subplots(2, len(LOG_COLS), figsize=(26, 6.5))
for j, col in enumerate(LOG_COLS):
    axes[0, j].hist(df[col].dropna(), bins=50, color="#C44E52")
    axes[0, j].set_title(f"{col}\n(mentah)", fontsize=9)
    axes[1, j].hist(np.log1p(df[col].dropna()), bins=50, color="#4C72B0")
    axes[1, j].set_title("log1p", fontsize=9)
plt.suptitle("Efek transformasi log1p pada kolom uang/jumlah", y=1.02)
plt.tight_layout()
plt.show()

> **🔄 Alur data: Fitur Linear & Kategorikal**
>
> - **Input:** `df`: 4 kolom linear, `preferred_channel`, 3 kolom profil.
> - **Proses:**
>   1. Histogram `age`, `tenure_years`, `num_products`, `digital_txn_ratio`.
>   2. Bar chart frekuensi kanal, pekerjaan, status karyawan, dan tier kota.
> - **Output:** Grafik. Umur 22–65 (median 38), lama nasabah median 5,1 tahun, median 3 produk, rasio digital median 0,70. Kanal: mobile 49%, cabang 19%, ATM 18%, internet banking 14%.
> - **Berikutnya:** Cek korelasi antar fitur numerik.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 8))
for ax, col in zip(axes[0], LINEAR_COLS):
    ax.hist(df[col], bins=40, color="#55A868")
    ax.set_title(col)
for ax, col in zip(axes[1], CAT_COLS + PROFILE_CAT_COLS):
    df[col].value_counts().sort_values().plot.barh(ax=ax, color="#8172B2")
    ax.set_title(col)
plt.tight_layout()
plt.show()

> **🔄 Alur data: Korelasi Spearman**
>
> - **Input:** `df[NUMERIC_COLS]`.
> - **Proses:**
>   1. Korelasi Spearman (berbasis peringkat, cocok untuk data miring) antar 11 fitur numerik.
> - **Output:** Heatmap. Korelasi terkuat: gaji ↔ nominal transaksi (0,89), gaji ↔ saldo (0,79), saldo ↔ nominal transaksi (0,75). Rasio digital berkorelasi negatif dengan umur (−0,36).
> - **Berikutnya:** EDA selesai; data dibagi berdasarkan waktu.

In [ ]:
corr = df[NUMERIC_COLS].corr(method="spearman")
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Korelasi Spearman antar fitur numerik")
plt.tight_layout()
plt.show()

### Insight EDA

- **Kualitas data baik**: tidak ada duplikat, dan satu-satunya missing adalah `monthly_salary` (3,7%, gaji tidak dilaporkan). Missing ini diisi median di pipeline.
- **Banyak nilai nol**: 47% nasabah tidak memakai kartu kredit (`credit_card_spend` = 0), 15% tanpa pinjaman, dan 10% tanpa investasi. Nol di sini adalah **informasi** (bukan data hilang), jadi dibiarkan. Inilah alasan GMM perlu `reg_covar` (Bagian 7).
- **Kolom uang sangat miring**: skewness `investment_balance` 9,2, `avg_balance` 5,0, kartu kredit 4,5, dan nominal transaksi 4,4. Setelah `log1p` semuanya turun ke < 1,4. Tanpa log, segelintir nasabah dengan investasi miliaran akan menarik centroid dan mendominasi jarak.
- **Korelasi kuat antar fitur nilai**: gaji ↔ nominal transaksi (ρ = 0,89), gaji ↔ saldo (0,79), saldo ↔ nominal transaksi (0,75). Artinya ada satu sumbu besar “nilai/kemakmuran nasabah”, terlihat juga di PCA (PC1 menjelaskan 44% varians). Fitur-fitur ini tetap dipertahankan karena masing-masing membawa nuansa berbeda (mis. UMKM: transaksi besar tetapi investasi kecil).
- **Digital vs usia**: `digital_txn_ratio` berkorelasi negatif dengan umur (ρ = −0,36) dan lama menjadi nasabah (−0,26). Nasabah muda dan baru lebih digital.
- Separuh nasabah (49%) memilih **mobile** sebagai kanal utama, sisanya cabang (19%), ATM (18%), dan internet banking (14%).

## 4. Split Berdasarkan Waktu (Development vs Out-of-Time)

Data dibagi berdasarkan **bulan snapshot**, bukan acak:
- **dev (Jan–Sep 2025)**: untuk eksplorasi, preprocessing, pemilihan k, pemilihan model, dan training.
- **OOT (Okt–Des 2025)**: disimpan dan hanya dipakai di Bagian 13, untuk menguji apakah segmen yang ditemukan **masih berlaku** di periode berikutnya. Ini padanan *backtesting* untuk clustering.

> **🔄 Alur data: Split Waktu: dev vs OOT**
>
> - **Input:** `df` (12.000 baris).
> - **Proses:**
>   1. `dev` = snapshot Jan–Sep 2025; `oot` = Okt–Des 2025. Index di-reset.
>   2. Median fitur dev vs OOT ditampilkan untuk melihat pergeseran awal.
> - **Output:** `dev` **9.049** nasabah (2025-01 – 2025-09), `oot` **2.951** (2025-10 – 2025-12). Median rasio digital naik 0,682 → 0,768. Fitur lain relatif mirip.
> - **Berikutnya:** `oot` dikunci sampai Bagian 13. Preprocessing di-fit di `dev`.

In [ ]:
dev = df[df[DATE_COL] < OOT_START].reset_index(drop=True)
oot = df[df[DATE_COL] >= OOT_START].reset_index(drop=True)
print(f"dev : {len(dev):,} nasabah ({dev[DATE_COL].min()} – {dev[DATE_COL].max()})")
print(f"OOT : {len(oot):,} nasabah ({oot[DATE_COL].min()} – {oot[DATE_COL].max()})")
pd.DataFrame({"dev": dev[NUMERIC_COLS].median(), "OOT": oot[NUMERIC_COLS].median()}).round(3).T

## 5. Preprocessing

Clustering berbasis **jarak**, jadi skala fitur sangat menentukan hasil. Preprocessing di `src/features.py`:

```
log_num : 7 kolom uang/jumlah → isi NaN (median) → log1p → StandardScaler
num     : age, tenure_years, num_products, digital_txn_ratio → isi NaN (median) → StandardScaler
cat     : preferred_channel → one-hot → × bobot 0,5
```

| Langkah | Alasan |
|---|---|
| **log1p** | Saldo, investasi, dan pinjaman sangat miring (skewness > 3). Tanpa log, beberapa nasabah sangat kaya akan menjadi “cluster” sendiri dan jarak didominasi nominal rupiah. log1p juga aman untuk nilai 0. |
| **StandardScaler** | Semua fitur jadi mean 0 & std 1, sehingga umur (tahun) dan saldo (juta) punya pengaruh setara. |
| **One-hot × 0,5** | Dua nasabah dengan kanal berbeda berjarak² 2 × 0,5² = 0,5 (setara selisih ±0,7 std di satu fitur numerik). Kanal ikut berpengaruh tanpa mendominasi 11 fitur numerik. |

**Kenapa binning tidak dipakai sebagai input model?** Binning mengubah angka menjadi kategori. Dalam jarak Euclidean, one-hot dari bin tidak punya urutan: umur 26–35 sama jauhnya ke 36–45 dan ke 56+. Untuk clustering, nilai kontinu yang di-scale lebih tepat. Binning (`age_band`, `monthly_salary_band`) dipakai di Bagian 11 untuk **membaca profil segmen** dalam bahasa bisnis.

> **🔄 Alur data: Fit Preprocessor**
>
> - **Input:** `dev[FEATURES]` (12 kolom mentah).
> - **Proses:**
>   1. `fit` mempelajari dari dev: median (pengisi NaN gaji), mean & std setelah log1p, mean & std kolom linear, daftar kategori kanal.
>   2. `transform` menerapkan: isi NaN → log1p (7 kolom) → scale → one-hot kanal × 0,5.
> - **Output:** `preprocessor` (sudah di-fit), `X_dev` (**9.049 × 15**: 7 log + 4 linear + 4 one-hot kanal), `feature_names_out`. Semua fitur numerik mean 0 & std 1; kolom kanal bernilai 0 atau 0,5.
> - **Berikutnya:** Telusuri satu nasabah.

In [ ]:
preprocessor = build_preprocessor()
X_dev = preprocessor.fit_transform(dev[FEATURES])
feature_names_out = preprocessor.get_feature_names_out()
print(f"{len(FEATURES)} kolom mentah → {X_dev.shape[1]} kolom setelah preprocessing | X_dev: {X_dev.shape}")
pd.DataFrame(X_dev, columns=feature_names_out).describe().T[["mean", "std", "min", "max"]].round(3)

> **🔄 Alur data: Telusuri 1 Nasabah**
>
> - **Input:** Baris pertama `dev` dan baris pertama `X_dev`.
> - **Proses:**
>   1. Menampilkan 12 nilai mentah dan 15 nilai hasil preprocessing.
> - **Output:** Contoh nasabah 29 tahun, gaji 19,2 jt, rasio digital 0,97, kanal mobile → `monthly_salary` +1,01 std, `digital_txn_ratio` +1,33 std, `age` −0,98 std, `tenure_years` −1,29 std, `preferred_channel_mobile` = 0,5. Profil khas Profesional Muda Digital.
> - **Berikutnya:** Uji apakah data punya struktur cluster.

In [ ]:
contoh = dev.iloc[[0]]
trace = pd.DataFrame({
    "nilai_mentah": contoh[FEATURES].T.iloc[:, 0],
})
z = pd.Series(X_dev[0], index=feature_names_out)
display(trace)
print("Setelah preprocessing (15 angka yang dilihat model):")
display(z.to_frame("nilai_transformasi").round(4))

## 6. Uji Kecenderungan Cluster (Cluster Tendency)

Algoritma clustering **selalu** menghasilkan cluster, bahkan dari data acak. Jadi sebelum mencari segmen, perlu diuji dulu apakah data **memang punya struktur kelompok**.

**Hopkins statistic (H):** jarak tetangga terdekat dari titik acak seragam dibandingkan dengan jarak dari titik data asli. H ≈ 0,5 berarti data tersebar acak, H > 0,7 berarti ada kecenderungan cluster yang kuat. Sebagai pembanding, H juga dihitung pada **data nol** (setiap kolom diacak sendiri-sendiri, sehingga distribusi tiap fitur sama tetapi hubungan antar fitur hilang).

> **🔄 Alur data: Hopkins Statistic & PCA**
>
> - **Input:** `X_dev`.
> - **Proses:**
>   1. Hopkins dihitung pada data asli dan pada 20 versi data nol (`permute_columns`: tiap kolom diacak sendiri).
>   2. PCA 2 komponen di-fit di `X_dev` (`pca_dev`, dipakai lagi di Bagian 11). Plot kepadatan data asli vs data nol.
> - **Output:** `hopkins_real` = **0,789** vs data nol 0,658 ± 0,003. `pca_dev` (2 komponen menjelaskan 61% varians: PC1 44%, PC2 17%). Peta kepadatan memperlihatkan beberapa “pulau”, sedangkan data nol berbentuk satu gumpalan.
> - **Berikutnya:** Definisikan 2 model kandidat.

In [ ]:
hopkins_real = hopkins_statistic(X_dev, seed=RANDOM_STATE)
hopkins_null = np.array([hopkins_statistic(permute_columns(X_dev, np.random.default_rng(s)), seed=s) for s in range(20)])
print(f"Hopkins data asli : {hopkins_real:.3f}")
print(f"Hopkins data nol  : {hopkins_null.mean():.3f} ± {hopkins_null.std(ddof=1):.3f} (20 permutasi)")

pca_dev = PCA(n_components=2, random_state=RANDOM_STATE).fit(X_dev)
P = pca_dev.transform(X_dev)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
axes[0].hexbin(P[:, 0], P[:, 1], gridsize=60, cmap="viridis", mincnt=1)
axes[0].set_title(f"Kepadatan data dev di 2 komponen PCA (varians terjelaskan {pca_dev.explained_variance_ratio_.sum():.0%})")
Pn = pca_dev.transform(permute_columns(X_dev, np.random.default_rng(0)))
axes[1].hexbin(Pn[:, 0], Pn[:, 1], gridsize=60, cmap="viridis", mincnt=1)
axes[1].set_title("Data nol (kolom diacak): tanpa struktur kelompok")
plt.tight_layout()
plt.show()

## 7. Dua Model Kandidat

| | **K-Means** | **Gaussian Mixture (GMM)** |
|---|---|---|
| Ide | Setiap nasabah masuk ke centroid terdekat | Data dianggap campuran beberapa distribusi normal |
| Bentuk cluster | Bulat, ukuran mirip | Elips, ukuran & orientasi bebas (covariance `full`) |
| Keanggotaan | Keras (1 segmen) | Lunak: probabilitas masuk ke tiap segmen |
| Pemilihan k | Elbow, silhouette | + **BIC/AIC** (berbasis likelihood) |
| Kelebihan | Sederhana, cepat, mudah dijelaskan | Fleksibel, memberi tingkat keyakinan |
| Kekurangan | Asumsi bentuk bulat | Lebih banyak parameter, bisa degenerasi |

**Catatan teknis GMM: `reg_covar`.** Beberapa fitur punya banyak nilai persis sama (belanja kartu kredit = 0 pada nasabah tanpa kartu, investasi = 0, one-hot kanal hanya 0/0,5). GMM dengan covariance penuh bisa “curang” dengan membuat komponen yang sangat sempit di titik-titik tersebut. Likelihood-nya jadi sangat besar, sehingga **BIC menyesatkan**. Cell berikutnya membuktikannya. Solusinya adalah regularisasi `reg_covar = 0,05` (variance minimum 0,05 di setiap dimensi, kecil dibanding variance fitur yang sudah di-scale = 1).

> **🔄 Alur data: Definisi Model**
>
> - **Input:** Tidak ada data.
> - **Proses:**
>   1. `MODEL_FACTORIES`: fungsi pembuat model untuk k tertentu, yaitu K-Means (`n_init=10`) dan GMM (`full`, `reg_covar=0,05`, `n_init=3`). Memakai *factory* agar setiap uji membuat model baru yang belum dilatih.
> - **Output:** `MODEL_FACTORIES`, `MODEL_LABELS`.
> - **Berikutnya:** Buktikan perlunya `reg_covar`.

In [ ]:
MODEL_FACTORIES = {
    "kmeans": lambda k: KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE),
    "gmm": lambda k: GaussianMixture(n_components=k, covariance_type="full", reg_covar=0.05,
                                     n_init=3, random_state=RANDOM_STATE),
}
MODEL_LABELS = {"kmeans": "K-Means", "gmm": "Gaussian Mixture"}

> **🔄 Alur data: Demonstrasi reg_covar GMM**
>
> - **Input:** `X_dev`.
> - **Proses:**
>   1. GMM dilatih untuk k = 2…10 dengan `reg_covar` 1e-6 (default) dan 0,05.
>   2. BIC dan silhouette dibandingkan.
> - **Output:** `reg_demo`. reg_covar 1e-6: BIC melonjak tidak wajar di k = 4, 6, 8, 9 (hingga −174.653) dengan silhouette 0,05–0,10. reg_covar 0,05: BIC turun mulus dan mulai mendatar setelah k = 5; silhouette tertinggi juga di k = 5 (0,305).
> - **Berikutnya:** Scan k untuk kedua model.

In [ ]:
rows = []
for reg in [1e-6, 0.05]:
    for k in K_RANGE:
        gm = GaussianMixture(k, covariance_type="full", reg_covar=reg, n_init=3, random_state=RANDOM_STATE).fit(X_dev)
        lab = gm.predict(X_dev)
        rows.append({"reg_covar": reg, "k": k, "BIC": gm.bic(X_dev),
                     "silhouette": internal_metrics(X_dev, lab, SIL_SAMPLE)["silhouette"]})
reg_demo = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for reg, g in reg_demo.groupby("reg_covar"):
    axes[0].plot(g["k"], g["BIC"], "o-", label=f"reg_covar={reg:g}")
    axes[1].plot(g["k"], g["silhouette"], "o-", label=f"reg_covar={reg:g}")
axes[0].set_title("BIC GMM (makin kecil = makin baik)")
axes[1].set_title("Silhouette dari label GMM")
for ax in axes:
    ax.set_xlabel("k")
    ax.legend()
plt.tight_layout()
plt.show()
reg_demo.pivot(index="k", columns="reg_covar", values=["BIC", "silhouette"]).round(3)

## 8. Pencarian Jumlah Cluster (k)

Untuk setiap model dan setiap k = 2…10 dihitung:

| Kriteria | Arah | Arti |
|---|---|---|
| **Inertia** (K-Means) | ↓, cari “siku” | jumlah jarak² ke centroid |
| **BIC / AIC** (GMM) | ↓ | likelihood dikurangi penalti jumlah parameter |
| **Silhouette** | ↑ (−1…1) | seberapa dekat nasabah ke segmennya dibanding ke segmen terdekat lain |
| **Calinski-Harabasz** | ↑ | rasio sebaran antar-cluster vs dalam-cluster |
| **Davies-Bouldin** | ↓ | rata-rata kemiripan tiap cluster dengan cluster paling mirip |
| **Stabilitas bootstrap** (ARI, 10×) | ↑ | apakah cluster yang sama muncul lagi jika data di-resample |
| **Segmen terkecil** | — | porsi nasabah di segmen terkecil (segmen < 3% sulit ditindaklanjuti) |

**Aturan pemilihan k (ditetapkan sebelum melihat hasil):**
1. k harus di rentang bisnis **4–8** segmen. Kurang dari 4 terlalu kasar untuk strategi produk, lebih dari 8 sulit dikelola tim marketing.
2. Stabilitas bootstrap harus **≥ 0,75** (ambang “stabil” menurut Hennig, 2007).
3. Dari yang lolos, pilih **silhouette tertinggi**. Kriteria lain dipakai sebagai bukti pendukung.

> **🔄 Alur data: Scan k = 2…10**
>
> - **Input:** `X_dev`, `MODEL_FACTORIES`.
> - **Proses:**
>   1. Untuk 2 model × 9 nilai k: fit → label → silhouette (sampel 5.000), Calinski-Harabasz, Davies-Bouldin.
>   2. Stabilitas bootstrap 10× (`bootstrap_stability`): model dilatih di sampel bootstrap, memprediksi seluruh dev, lalu ARI dihitung terhadap label referensi.
>   3. Porsi segmen terkecil, inertia (K-Means), BIC/AIC (GMM).
> - **Output:** `scan` (18 baris). K-Means: silhouette tertinggi di k = 2 (0,346), di rentang 4–8 tertinggi di k = 5 (0,311); stabilitas 0,99 untuk k ≤ 6, turun ke ±0,92 untuk k ≥ 7. GMM: silhouette tertinggi di k = 5 (0,305), stabilitas k = 5 = 0,965.
> - **Berikutnya:** Visualisasikan semua kriteria.

In [ ]:
scan_rows = []
for name, factory in MODEL_FACTORIES.items():
    for k in K_RANGE:
        model = factory(k).fit(X_dev)
        labels = model.predict(X_dev)
        row = {"model": name, "k": k, **internal_metrics(X_dev, labels, SIL_SAMPLE, RANDOM_STATE),
               "stability_ari": bootstrap_stability(factory(k), X_dev, n_boot=N_BOOT_SCAN, seed=RANDOM_STATE)["ari_mean"],
               "min_segment_share": np.bincount(labels, minlength=k).min() / len(labels)}
        if name == "kmeans":
            row["inertia"] = model.inertia_
        else:
            row["BIC"], row["AIC"] = model.bic(X_dev), model.aic(X_dev)
        scan_rows.append(row)
scan = pd.DataFrame(scan_rows)
scan.round(4)

> **🔄 Alur data: Grafik Pencarian k**
>
> - **Input:** `scan`.
> - **Proses:**
>   1. 6 panel: elbow inertia, BIC/AIC, silhouette, CH, DB, stabilitas. Area abu-abu = rentang bisnis 4–8.
> - **Output:** Grafik 6 panel. Siku inertia di sekitar k = 5; BIC mendatar setelah k = 5; silhouette kedua model memuncak di k = 5 dalam rentang bisnis; stabilitas mulai turun di k ≥ 6–7.
> - **Berikutnya:** Terapkan aturan pemilihan k.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 9))
km, gm = scan[scan.model == "kmeans"], scan[scan.model == "gmm"]
axes[0, 0].plot(km["k"], km["inertia"], "o-")
axes[0, 0].set_title("K-Means: inertia (elbow)")
axes[0, 1].plot(gm["k"], gm["BIC"], "o-", label="BIC")
axes[0, 1].plot(gm["k"], gm["AIC"], "s--", label="AIC")
axes[0, 1].set_title("GMM: BIC & AIC")
axes[0, 1].legend()
for ax, metric, title in [(axes[0, 2], "silhouette", "Silhouette ↑"), (axes[1, 0], "calinski_harabasz", "Calinski-Harabasz ↑"),
                          (axes[1, 1], "davies_bouldin", "Davies-Bouldin ↓"), (axes[1, 2], "stability_ari", "Stabilitas bootstrap (ARI) ↑")]:
    for name, g in scan.groupby("model"):
        ax.plot(g["k"], g[metric], "o-", label=MODEL_LABELS[name])
    ax.set_title(title)
    ax.legend()
axes[1, 2].axhline(STABILITY_MIN, color="k", ls="--", lw=1)
for ax in axes.ravel():
    ax.axvspan(BUSINESS_K[0] - 0.5, BUSINESS_K[1] + 0.5, color="grey", alpha=0.1)
    ax.set_xlabel("k")
plt.suptitle("Pencarian k (area abu-abu = rentang bisnis)", y=1.01)
plt.tight_layout()
plt.show()

> **🔄 Alur data: Aturan Pemilihan k**
>
> - **Input:** `scan`, `X_dev`, `dev`.
> - **Proses:**
>   1. `choose_k`: filter k di rentang bisnis & ARI bootstrap ≥ 0,75, lalu ambil silhouette tertinggi.
>   2. Untuk transparansi: tampilkan silhouette tertinggi di seluruh rentang dan profil K-Means k = 2 (kenapa k = 2 tidak dipilih).
> - **Output:** `BEST_K = {kmeans: 5, gmm: 5}`. K-Means k = 2 hanya memisahkan nasabah “bernilai rendah” (6.788; gaji 6,6 jt) vs “bernilai tinggi” (2.261; gaji 24 jt), terlalu kasar untuk strategi produk.
> - **Berikutnya:** Bangun 2 kandidat final dan uji lebih ketat.

In [ ]:
def choose_k(scan_model: pd.DataFrame) -> int:
    ok = scan_model[scan_model["k"].between(*BUSINESS_K) & (scan_model["stability_ari"] >= STABILITY_MIN)]
    return int(ok.loc[ok["silhouette"].idxmax(), "k"])


BEST_K = {name: choose_k(scan[scan.model == name]) for name in MODEL_FACTORIES}
best_overall_sil = scan.loc[scan["silhouette"].idxmax()]
print("k terpilih per model:", BEST_K)
print(f"Catatan: silhouette tertinggi di SELURUH rentang = {best_overall_sil['silhouette']:.3f} "
      f"({MODEL_LABELS[best_overall_sil['model']]}, k={int(best_overall_sil['k'])})")

# kenapa bukan k=2? lihat apa yang dipisahkan oleh 2 cluster
k2 = MODEL_FACTORIES["kmeans"](2).fit(X_dev).predict(X_dev)
print("\nProfil K-Means k=2 (median):")
display(dev.assign(cluster=k2).groupby("cluster")[NUMERIC_COLS].median().round(2).assign(n=np.bincount(k2)))

### 8.1 Model Statistik Kandidat: VIF, Intercept, Koefisien & p-value

Sebelum diuji stabilitasnya (Bagian 9, padanan cross-validation), kandidat K-Means dengan k terpilih dipahami dulu secara statistik:

1. **VIF fitur input**: clustering memakai jarak Euclidean. Fitur yang saling berkorelasi tinggi pada dasarnya mengukur hal yang sama, sehingga dimensi itu **terhitung beberapa kali** dalam jarak dan mendapat bobot lebih besar dari seharusnya. VIF menunjukkan dimensi mana yang “dobel”.
2. **Regresi OLS per fitur terhadap segmen**: fitur (terstandar) ~ intercept + dummy segmen, dengan segmen terbesar sebagai **acuan**:
   - **intercept** = rata-rata fitur di segmen acuan (satuan std; 0 = rata-rata semua nasabah),
   - **koefisien** segmen = selisih rata-rata segmen itu terhadap segmen acuan (satuan std),
   - **p-value** dengan standard error robust **HC3** (varians antar segmen tidak harus sama),
   - **R² (η²)** = porsi variasi fitur yang dijelaskan oleh pembagian segmen.

**Kenapa bukan regresi logistik multinomial (segmen ~ fitur)?** Batas antar cluster K-Means bersifat linear (sel Voronoi), sehingga logit multinomial akan **memisahkan segmen secara sempurna**. Akibatnya koefisien menuju tak hingga dan p-value tidak valid. Arah sebaliknya, fitur ~ segmen, selalu terdefinisi.

> ⚠️ **Catatan penting**: segmen dibentuk **dari fitur-fitur yang sama**, jadi p-value di sini pasti sangat kecil (*circular*). p-value ini **mendeskripsikan** perbedaan antar segmen, bukan **membuktikan** bahwa segmen itu nyata. Bukti bahwa struktur segmen nyata dan stabil ada di uji permutasi & bootstrap (Bagian 9). Yang paling informatif di sini adalah **besar koefisien** dan **R²**.

> **🔄 Alur data: VIF Fitur Input Clustering**
>
> - **Input:** `X_dev` (9.049 × 15, sudah log & scaling).
> - **Proses:**
>   1. One-hot kanal: 1 kategori acuan dibuang (kolom kanal selalu berjumlah 0,5 → kolinear dengan intercept).
>   2. VIF setiap fitur terhadap 13 fitur lain + intercept; status aman/sedang/tinggi.
>   3. 5 pasangan fitur numerik dengan korelasi tertinggi.
> - **Output:** `vif_clu`: rank 15/15 (acuan kanal = *mobile*); **tidak ada VIF > 10**. Sedang: nominal transaksi 6,5, gaji 6,0, saldo 5,2. Korelasi tertinggi: gaji–saldo 0,83, gaji–nominal transaksi 0,83, saldo–nominal transaksi 0,81.
> - **Berikutnya:** Regresi fitur terhadap segmen kandidat.

In [ ]:
Xd = pd.DataFrame(X_dev, columns=[c.split("__", 1)[1] for c in feature_names_out])
channel_cols = [c for c in Xd.columns if c.startswith("preferred_channel_")]
channel_ref = Xd[channel_cols].sum().idxmax()
Xv = Xd.drop(columns=channel_ref)          # one-hot kanal: buang 1 kategori acuan (jumlahnya konstan → kolinear dengan intercept)
A_vif = sm.add_constant(Xv).to_numpy(float)
vif_clu = pd.DataFrame({"VIF": [variance_inflation_factor(A_vif, i + 1) for i in range(Xv.shape[1])]}, index=Xv.columns)
vif_clu["status"] = pd.cut(vif_clu["VIF"], [0, 5, 10, np.inf], labels=["aman", "sedang", "tinggi"])
corr_pairs = Xd[NUMERIC_COLS].corr().where(np.triu(np.ones((len(NUMERIC_COLS),) * 2, bool), 1)).stack().sort_values(key=abs, ascending=False)
print(f"Kategori acuan kanal: {channel_ref} | rank matriks {np.linalg.matrix_rank(A_vif)} dari {A_vif.shape[1]} kolom")
display(vif_clu.sort_values("VIF", ascending=False).round(2))
print("5 pasangan fitur (setelah log & scaling) dengan korelasi tertinggi:")
display(corr_pairs.head(5).round(3).to_frame("korelasi"))

> **🔄 Alur data: OLS Fitur ~ Segmen (Intercept, Koefisien, p-value)**
>
> - **Input:** `X_dev`, kandidat K-Means k terpilih, aturan persona (pratinjau nama).
> - **Proses:**
>   1. Label segmen kandidat → dummy (segmen terbesar = acuan).
>   2. Untuk 11 fitur numerik: OLS fitur ~ intercept + dummy segmen, SE robust HC3 → koefisien, p-value, CI 95%, R² (η²), F-test robust.
>   3. p-value semua koefisien dikoreksi Holm; heatmap intercept + koefisien dengan bintang signifikansi.
> - **Output:** `seg_coef` (55 baris), `seg_fit`, heatmap. Acuan **Massal Penabung** (35%). R² 0,36–**0,86** (tertinggi nominal transaksi). Contoh: Affluent investasi +2,81 SD; UMKM frekuensi +2,56; Bergantung Kredit pinjaman +1,68 & saldo −0,19; Muda Digital rasio digital +1,94 & umur −1,28. Semua signifikan setelah Holm (circular).
> - **Berikutnya:** Uji stabilitas mendalam (Bagian 9).

In [ ]:
cand_k = BEST_K["kmeans"]
cand_labels = MODEL_FACTORIES["kmeans"](cand_k).fit(X_dev).predict(X_dev)
preview_names = name_segments(dev[NUMERIC_COLS].assign(c=cand_labels).groupby("c").median())
seg_code_preview = pd.Series(cand_labels).map({c: s["code"] for c, s in preview_names.items()})
REF_SEGMENT = seg_code_preview.value_counts().idxmax()
seg_dummies = pd.get_dummies(seg_code_preview, dtype=float).drop(columns=REF_SEGMENT)
X_seg = sm.add_constant(seg_dummies)

coef_rows, fit_rows = [], []
for col in NUMERIC_COLS:
    ols = sm.OLS(Xd[col].to_numpy(), X_seg).fit(cov_type="HC3")
    ci = ols.conf_int()
    for term in ols.params.index:
        coef_rows.append({"fitur": col, "term": "INTERCEPT (" + REF_SEGMENT + ")" if term == "const" else term,
                          "koefisien": ols.params[term], "std_error_HC3": ols.bse[term], "p_value": ols.pvalues[term],
                          "ci95_lower": ci.loc[term, 0], "ci95_upper": ci.loc[term, 1]})
    fit_rows.append({"fitur": col, "R2_eta2": ols.rsquared, "F_robust": float(ols.fvalue), "p_F": float(ols.f_pvalue)})
seg_coef = pd.DataFrame(coef_rows)
seg_coef["p_holm"] = holm_correction(seg_coef["p_value"])
seg_coef["signifikan"] = np.select([seg_coef["p_holm"] < 0.001, seg_coef["p_holm"] < 0.01, seg_coef["p_holm"] < 0.05], ["***", "**", "*"], default="")
seg_fit = pd.DataFrame(fit_rows).set_index("fitur").sort_values("R2_eta2", ascending=False)

print(f"Kandidat K-Means k = {cand_k}; segmen acuan = {REF_SEGMENT} (terbesar, {(seg_code_preview == REF_SEGMENT).mean():.0%} nasabah)")
display(seg_fit.round(4))
wide = seg_coef.pivot(index="fitur", columns="term", values="koefisien").loc[seg_fit.index]
stars = seg_coef.pivot(index="fitur", columns="term", values="signifikan").loc[seg_fit.index]
annot = wide.round(2).astype(str) + stars
fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(wide, annot=annot, fmt="", cmap="RdBu_r", center=0, ax=ax, cbar_kws={"label": "satuan std"})
ax.set_title(f"Intercept (rata-rata segmen acuan) & koefisien segmen vs acuan — OLS, SE HC3, * p_holm<0,05")
plt.tight_layout()
plt.show()
with pd.option_context("display.max_rows", 80, "display.float_format", "{:.4f}".format):
    display(seg_coef.set_index(["fitur", "term"]).drop(columns=["ci95_lower", "ci95_upper"]))

#### Uji Homogenitas Varians & Perbandingan Berpasangan Semua Segmen

- **Brown-Forsythe** (Levene berbasis median): H0 = varians fitur sama di semua segmen. Jika ditolak, uji yang mengasumsikan varians sama (ANOVA klasik, Tukey HSD) tidak tepat, sehingga dipakai SE robust HC3 (di atas) dan **Games-Howell** (di bawah).
- **Games-Howell**: membandingkan **setiap pasangan** segmen (10 pasangan untuk 5 segmen), bukan hanya terhadap acuan. Family-wise error terkontrol lewat distribusi studentized range. Yang menarik justru pasangan yang **tidak** berbeda: di dimensi itu, kedua segmen serupa.

> **🔄 Alur data: Brown-Forsythe & Games-Howell**
>
> - **Input:** `Xd` (fitur terstandar), `seg_code_preview`.
> - **Proses:**
>   1. Brown-Forsythe per fitur (H0: varians sama di 5 segmen) + rasio SD maksimum/minimum.
>   2. Games-Howell untuk 10 pasangan segmen × 11 fitur = 110 perbandingan, p-value Holm.
>   3. Ringkasan: berapa pasangan yang berbeda signifikan per fitur, dan pasangan mana yang TIDAK berbeda.
> - **Output:** ⟪B_GH⟫
> - **Berikutnya:** Cari dimensi pemisah utama dengan LDA.

In [ ]:
lev_rows = []
for col in NUMERIC_COLS:
    parts = [Xd.loc[seg_code_preview == s, col] for s in sorted(seg_code_preview.unique())]
    stat, p = stats.levene(*parts, center="median")
    sds = [x.std() for x in parts]
    lev_rows.append({"fitur": col, "brown_forsythe": stat, "p_value": p, "rasio_SD_maks/min": max(sds) / min(sds)})
levene_table = pd.DataFrame(lev_rows).set_index("fitur")
levene_table["p_holm"] = holm_correction(levene_table["p_value"])
display(levene_table.round(4))

gh = pd.concat({col: games_howell(Xd[col], seg_code_preview) for col in NUMERIC_COLS}, names=["fitur"]).reset_index(level=0).reset_index(drop=True)
gh["p_holm"] = holm_correction(gh["p_value"])
gh["pasangan"] = gh["grup_a"] + " vs " + gh["grup_b"]
gh_summary = gh.groupby("fitur").apply(lambda g: pd.Series({"pasangan_berbeda_signifikan": int((g["p_holm"] < 0.05).sum()),
                                                            "dari": len(g),
                                                            "pasangan_TIDAK_berbeda": ", ".join(g.loc[g["p_holm"] >= 0.05, "pasangan"]) or "-"}),
                                     include_groups=False)
display(gh_summary)
print(f"Total: {(gh['p_holm'] < 0.05).sum()} dari {len(gh)} perbandingan berpasangan signifikan (Holm)")

#### Linear Discriminant Analysis (LDA): Dimensi Apa yang Memisahkan Segmen?

LDA mencari kombinasi linear fitur (**fungsi diskriminan**) yang paling memisahkan segmen. Ada tiga hal yang diperiksa:
- **Proporsi pemisahan** tiap fungsi diskriminan (mirip “varians terjelaskan” di PCA, tetapi untuk perbedaan antar segmen).
- **Structure matrix**: korelasi tiap fitur dengan skor fungsi diskriminan. Ini lebih stabil daripada koefisien LDA saat fitur berkorelasi, sehingga dipakai untuk menamai dimensi.
- **Akurasi klasifikasi ulang (CV 5-fold)**: seberapa “linear” batas antar segmen.

> **🔄 Alur data: Linear Discriminant Analysis**
>
> - **Input:** `Xd[NUMERIC_COLS]`, `seg_code_preview`.
> - **Proses:**
>   1. LDA → 4 fungsi diskriminan (jumlah segmen − 1), proporsi pemisahan tiap fungsi.
>   2. Structure matrix: korelasi setiap fitur dengan skor LD.
>   3. Akurasi klasifikasi ulang segmen dengan LDA (CV 5-fold).
> - **Output:** ⟪B_LDA⟫
> - **Berikutnya:** Interpretasi, lalu uji stabilitas (Bagian 9).

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score

lda_X = Xd[NUMERIC_COLS]
lda = LinearDiscriminantAnalysis().fit(lda_X, seg_code_preview)
lda_scores = lda.transform(lda_X)
n_ld = lda_scores.shape[1]
structure = pd.DataFrame({f"LD{i + 1}": [np.corrcoef(lda_X[c], lda_scores[:, i])[0, 1] for c in NUMERIC_COLS] for i in range(n_ld)}, index=NUMERIC_COLS)
lda_cv_acc = cross_val_score(LinearDiscriminantAnalysis(), lda_X, seg_code_preview, cv=5)
print("Proporsi pemisahan per fungsi diskriminan:", {f"LD{i + 1}": round(v, 3) for i, v in enumerate(lda.explained_variance_ratio_)})
print(f"Akurasi klasifikasi ulang LDA (CV 5-fold): {lda_cv_acc.mean():.3f} ± {lda_cv_acc.std():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.heatmap(structure, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=axes[0])
axes[0].set_title("Structure matrix: korelasi fitur dengan fungsi diskriminan")
for s in sorted(seg_code_preview.unique()):
    m = (seg_code_preview == s).to_numpy()
    axes[1].scatter(lda_scores[m, 0], lda_scores[m, 1], s=4, alpha=0.4, label=s)
axes[1].set_xlabel(f"LD1 ({lda.explained_variance_ratio_[0]:.0%})")
axes[1].set_ylabel(f"LD2 ({lda.explained_variance_ratio_[1]:.0%})")
axes[1].set_title("Segmen kandidat di ruang diskriminan")
axes[1].legend(markerscale=4)
plt.tight_layout()
plt.show()

#### Interpretasi Model Statistik Kandidat

**1. VIF: dimensi “nilai nasabah” terhitung lebih dari sekali**
- Tidak ada kolinearitas sempurna (rank 15 dari 15 kolom setelah kanal *mobile* dijadikan acuan), dan **tidak ada VIF > 10**.
- Tiga fitur berstatus **sedang**: nominal transaksi (VIF 6,5), gaji (6,0), dan saldo (5,2). Ketiganya saling berkorelasi 0,81–0,83 setelah log & scaling, ditambah investasi (0,77–0,80). Dalam jarak Euclidean, “seberapa kaya nasabah” praktis **terhitung 3–4 kali**, dan inilah alasan PC1 menjelaskan 44% varians.
- **Dampaknya:** segmentasi condong membedakan nasabah berdasarkan **nilai/kemakmuran**. Untuk tujuan bisnis (strategi per nilai nasabah) hal ini **masih bisa diterima**. Jika ingin semua dimensi berbobot setara, opsinya adalah menggabungkan fitur berkorelasi (mis. PCA/whitening) atau membuang salah satunya, lalu ulangi seluruh evaluasi.

**2. Regresi fitur ~ segmen (acuan: Massal Penabung, 35% nasabah)**
- **Intercept** = posisi segmen acuan (satuan std; 0 = rata-rata semua nasabah). Massal Penabung berada **di bawah rata-rata** di semua dimensi nilai: gaji −0,88 SD, saldo −0,65, investasi −0,66, frekuensi transaksi −1,01, digital −0,84, produk −0,84. Hanya umur yang sedikit di atas rata-rata (+0,26).
- **Koefisien** = selisih tiap segmen terhadap Massal Penabung:
  - **Affluent**: investasi **+2,81 SD**, produk +2,80, gaji +2,75, saldo +2,50, lama nasabah +1,37, umur +0,94
  - **UMKM**: nominal transaksi **+2,59**, frekuensi +2,56, saldo +1,78, pinjaman +1,53
  - **Bergantung Kredit**: pinjaman **+1,68**, kartu kredit +1,67, frekuensi +1,06, tetapi saldo **−0,19** dan investasi −0,03. Saldo mereka bahkan **lebih rendah** dari Massal Penabung.
  - **Muda Digital**: rasio digital **+1,94**, frekuensi +1,69, umur **−1,28**, lama nasabah −0,86
- **R² (η²)**: pembagian 5 segmen menjelaskan **86%** variasi nominal transaksi, 81% saldo & frekuensi, 78% investasi, 75% gaji, dan paling sedikit lama menjadi nasabah (36%). Fitur dengan R² tinggi adalah **pembentuk utama** segmen.
- Semua 55 koefisien signifikan setelah Holm. Satu-satunya yang “hanya” `**` adalah umur UMKM vs Massal Penabung (−0,08 SD, p = 0,003): **signifikan tetapi praktis tidak ada bedanya**.

> ⚠️ Seperti dijelaskan di atas, p-value ini **circular**: segmen dibuat dari fitur yang sama, jadi wajar semuanya signifikan. Gunakan tabel ini untuk **membaca besarnya perbedaan**. Bukti bahwa segmen ini nyata dan stabil ada di Bagian 9 (uji permutasi p = 0,01, bootstrap ARI 0,996).

## 9. Evaluasi Mendalam Dua Kandidat (Padanan Cross-Validation)

Kedua kandidat (K-Means dan GMM, masing-masing dengan k terbaiknya) diuji lebih ketat:

| # | Uji | Pertanyaan |
|---|---|---|
| 9.1 | Metrik internal (silhouette **seluruh** data, CH, DB) + ukuran segmen | Seberapa kompak & terpisah? |
| 9.2 | **Silhouette per nasabah** | Berapa banyak nasabah yang “salah tempat” (silhouette < 0)? |
| 9.3 | **Bootstrap stability** (50×): ARI global + **Jaccard per cluster** | Apakah *setiap* segmen bisa direproduksi? |
| 9.4 | **Split-half “cross-validation”** (20×) | Dua model yang dilatih di dua separuh data berbeda: apakah hasilnya sama? |
| 9.5 | **Uji permutasi** (99×) | Apakah silhouette jauh lebih tinggi daripada data tanpa struktur? H0: tidak ada struktur |

**Aturan memilih model:** model yang menang di lebih banyak kriteria (silhouette, CH, DB, ARI bootstrap, Jaccard minimum, ARI split-half). Jika seri, pilih **K-Means** karena lebih sederhana dan mudah dijelaskan.

> **🔄 Alur data: Kandidat & Metrik Internal**
>
> - **Input:** `dev[FEATURES]`, `BEST_K`.
> - **Proses:**
>   1. Untuk tiap model: `Pipeline([preprocess, model(k terbaik)])` di-fit di dev → `candidates`.
>   2. Label dev → silhouette di **seluruh** data (bukan sampel), CH, DB, porsi segmen terkecil & terbesar.
> - **Output:** `candidates`, `cand_labels`, `cand_internal`. K-Means: silhouette **0,312**, CH 3.807, DB 1,287. GMM: 0,306, 3.763, 1,306. Segmen terkecil ±10%, terbesar ±35% (kedua model).
> - **Berikutnya:** Lihat silhouette per nasabah.

In [ ]:
candidates = {name: Pipeline([("preprocess", build_preprocessor()), ("model", MODEL_FACTORIES[name](BEST_K[name]))]).fit(dev[FEATURES])
              for name in MODEL_FACTORIES}
cand_labels = {name: pipe.predict(dev[FEATURES]) for name, pipe in candidates.items()}

rows = []
for name, labels in cand_labels.items():
    shares = np.bincount(labels) / len(labels)
    rows.append({"model": MODEL_LABELS[name], "k": BEST_K[name], **internal_metrics(X_dev, labels),
                 "segmen_terkecil": shares.min(), "segmen_terbesar": shares.max()})
cand_internal = pd.DataFrame(rows).set_index("model")
cand_internal.round(4)

> **🔄 Alur data: Diagram Silhouette**
>
> - **Input:** `X_dev`, `cand_labels`.
> - **Proses:**
>   1. `silhouette_samples` menghitung silhouette setiap nasabah, lalu nilainya diurutkan per cluster dan digambar sebagai “pisau”.
>   2. Persentase nasabah dengan silhouette < 0 (lebih dekat ke cluster lain) disimpan di `sil_neg`.
> - **Output:** Diagram silhouette. Nasabah dengan silhouette < 0: K-Means **1,5%**, GMM 3,1%.
> - **Berikutnya:** Uji stabilitas bootstrap 50×.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sil_neg = {}
for ax, (name, labels) in zip(axes, cand_labels.items()):
    s = silhouette_samples(X_dev, labels)
    sil_neg[name] = (s < 0).mean()
    y0 = 0
    for c in np.unique(labels):
        sc = np.sort(s[labels == c])
        ax.fill_betweenx(np.arange(y0, y0 + len(sc)), 0, sc, alpha=0.8)
        ax.text(-0.08, y0 + len(sc) / 2, str(c))
        y0 += len(sc) + 40
    ax.axvline(s.mean(), color="k", ls="--", label=f"rata-rata {s.mean():.3f}")
    ax.set_title(f"{MODEL_LABELS[name]} (k={BEST_K[name]}): {sil_neg[name]:.1%} nasabah silhouette < 0")
    ax.set_xlabel("silhouette")
    ax.legend()
plt.tight_layout()
plt.show()

> **🔄 Alur data: Bootstrap Stability (50×)**
>
> - **Input:** `X_dev`, model terbaik masing-masing.
> - **Proses:**
>   1. 50 sampel bootstrap → model dilatih ulang → memprediksi seluruh dev.
>   2. ARI global vs label referensi, dan **Jaccard per cluster**: untuk setiap cluster referensi dicari cluster bootstrap paling mirip (irisan/gabungan).
> - **Output:** `boot`, `boot_table`. K-Means: ARI **0,996 ± 0,002**, Jaccard cluster terlemah 0,994. GMM: ARI 0,982 ± 0,064, P5 0,818, Jaccard terlemah 0,952. Keduanya “stabil”.
> - **Berikutnya:** Uji reprodusibilitas dengan split-half.

In [ ]:
boot = {name: bootstrap_stability(MODEL_FACTORIES[name](BEST_K[name]), X_dev, n_boot=N_BOOT_FINAL, seed=RANDOM_STATE)
        for name in MODEL_FACTORIES}
rows = []
for name, b in boot.items():
    jac = b["jaccard_per_cluster"]
    rows.append({"model": MODEL_LABELS[name], "ARI_mean": b["ari_mean"], "ARI_std": b["ari_std"],
                 "ARI_p5": np.percentile(b["ari"], 5), "Jaccard_min_cluster": jac.mean().min(),
                 "Jaccard_mean": jac.mean().mean(), "status": stability_verdict(jac.mean().min())})
boot_table = pd.DataFrame(rows).set_index("model")
display(boot_table.round(4))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, (name, b) in zip(axes, boot.items()):
    b["jaccard_per_cluster"].boxplot(ax=ax)
    ax.axhline(STABILITY_MIN, color="r", ls="--", lw=1)
    ax.set_title(f"{MODEL_LABELS[name]}: Jaccard per cluster ({N_BOOT_FINAL} bootstrap)")
    ax.set_xlabel("cluster")
plt.tight_layout()
plt.show()

> **🔄 Alur data: Split-Half Cross-Validation (20×)**
>
> - **Input:** `X_dev`.
> - **Proses:**
>   1. 20× ulangan: dev dibagi dua separuh acak A & B → model A dan model B dilatih terpisah → keduanya melabeli seluruh dev → ARI antar pelabelan.
>   2. Mean, std, min, dan 95% CI (distribusi t). Wilcoxon membandingkan 20 pasangan skor kedua model.
> - **Output:** `split_half`, `split_table`. K-Means: ARI **0,993** (min 0,989). GMM: 0,981 (min **0,736**: sesekali GMM menemukan solusi lain). Wilcoxon p = 0,012.
> - **Berikutnya:** Uji apakah struktur lebih kuat dari kebetulan.

In [ ]:
split_half = {name: split_half_stability(MODEL_FACTORIES[name](BEST_K[name]), X_dev, n_repeats=N_SPLIT_HALF, seed=RANDOM_STATE)
              for name in MODEL_FACTORIES}
split_table = pd.DataFrame({
    MODEL_LABELS[n]: {"ARI_mean": v.mean(), "ARI_std": v.std(ddof=1), "ARI_min": v.min(),
                      "ci95_lower": v.mean() - stats.t.ppf(0.975, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v)),
                      "ci95_upper": v.mean() + stats.t.ppf(0.975, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v))}
    for n, v in split_half.items()}).T
w = stats.wilcoxon(split_half["kmeans"], split_half["gmm"])
display(split_table.round(4))
print(f"Wilcoxon signed-rank (K-Means vs GMM, {N_SPLIT_HALF} pasangan): p = {w.pvalue:.4f}")

> **🔄 Alur data: Uji Permutasi (99×)**
>
> - **Input:** `X_dev`.
> - **Proses:**
>   1. Silhouette data asli dihitung, lalu 99 data nol (kolom diacak) masing-masing di-cluster ulang dan dihitung silhouette-nya.
>   2. p-value = (jumlah data nol ≥ data asli + 1) / (99 + 1). z = (asli − rata-rata nol) / std nol.
> - **Output:** `perm`, `perm_table`. K-Means: silhouette 0,313 vs data nol 0,076 ± 0,002 (z = 104). GMM: 0,307 vs 0,049 (z = 22). Keduanya **p = 0,01** (minimum yang mungkin dengan 99 permutasi).
> - **Berikutnya:** Bandingkan semua kriteria & pilih model.

In [ ]:
perm = {name: permutation_test_silhouette(MODEL_FACTORIES[name](BEST_K[name]), X_dev, n_perm=N_PERM, seed=RANDOM_STATE)
        for name in MODEL_FACTORIES}
perm_table = pd.DataFrame({MODEL_LABELS[n]: {k: v for k, v in r.items() if k != "null"} for n, r in perm.items()}).T
display(perm_table.astype(float).round(4))

fig, ax = plt.subplots(figsize=(9, 4))
for name, r in perm.items():
    ax.hist(r["null"], bins=20, alpha=0.6, label=f"{MODEL_LABELS[name]}: data nol")
    ax.axvline(r["silhouette_real"], ls="--", lw=2, label=f"{MODEL_LABELS[name]}: data asli {r['silhouette_real']:.3f}")
ax.set_xlabel("silhouette")
ax.set_title(f"Uji permutasi ({N_PERM} data nol)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

> **🔄 Alur data: Perbandingan & Pemilihan Model**
>
> - **Input:** `cand_internal`, `boot`, `split_half`.
> - **Proses:**
>   1. 6 kriteria dibandingkan, dan pemenang tiap kriteria dicatat.
>   2. Model dengan kemenangan terbanyak dipilih (seri → K-Means).
> - **Output:** `comparison`, `wins`. **K-Means menang 6/6 kriteria** → `best_name = "kmeans"`, `other_name = "gmm"`.
> - **Berikutnya:** Model final dinamai segmennya.

In [ ]:
criteria = {  # (nilai per model, arah: True = makin besar makin baik)
    "silhouette": ({n: cand_internal.loc[MODEL_LABELS[n], "silhouette"] for n in MODEL_FACTORIES}, True),
    "calinski_harabasz": ({n: cand_internal.loc[MODEL_LABELS[n], "calinski_harabasz"] for n in MODEL_FACTORIES}, True),
    "davies_bouldin": ({n: cand_internal.loc[MODEL_LABELS[n], "davies_bouldin"] for n in MODEL_FACTORIES}, False),
    "bootstrap_ARI": ({n: boot[n]["ari_mean"] for n in MODEL_FACTORIES}, True),
    "Jaccard_min_cluster": ({n: boot[n]["jaccard_per_cluster"].mean().min() for n in MODEL_FACTORIES}, True),
    "split_half_ARI": ({n: split_half[n].mean() for n in MODEL_FACTORIES}, True),
}
rows = []
for crit, (vals, higher) in criteria.items():
    winner = (max if higher else min)(vals, key=vals.get)
    rows.append({"kriteria": crit, **{MODEL_LABELS[n]: v for n, v in vals.items()}, "pemenang": MODEL_LABELS[winner]})
comparison = pd.DataFrame(rows).set_index("kriteria")
wins = comparison["pemenang"].value_counts()
best_name = "kmeans" if wins.get("K-Means", 0) >= wins.get("Gaussian Mixture", 0) else "gmm"
other_name = "gmm" if best_name == "kmeans" else "kmeans"
display(comparison.round(4))
print(f"Jumlah kemenangan: {wins.to_dict()} → model terpilih: {MODEL_LABELS[best_name]} (k={BEST_K[best_name]})")

### Interpretasi Pemilihan k & Model

**Kecenderungan cluster (Bagian 6)**
- Hopkins data asli **0,789**, jauh di atas data nol (0,658 ± 0,003; 20 permutasi), jadi data **memang punya struktur kelompok**. Hopkins data nol tidak 0,5 karena distribusi fitur yang sudah miring/diskret ikut membuat titik acak seragam jauh dari data. Itulah kenapa pembanding data nol lebih jujur daripada patokan 0,5.

**Kenapa `reg_covar` penting (Bagian 7)**
- Tanpa regularisasi (`reg_covar` = 1e-6), BIC GMM “membaik” drastis di k = 4, 6, 8, 9 (BIC sampai −174.653), padahal silhouette-nya hancur (0,05–0,10). Komponen-komponen yang sangat sempit menempel di nilai nol/diskret. Dengan `reg_covar` = 0,05, BIC turun tajam sampai **k = 5** lalu mendatar, konsisten dengan silhouette. Jadi kriteria BIC **hanya bisa dipercaya** setelah model diregularisasi.

**Pemilihan k (Bagian 8)**
- Silhouette tertinggi di seluruh rentang ada di **K-Means k = 2 (0,346)**, tetapi k = 2 hanya memisahkan “nasabah bernilai rendah (75%)” vs “bernilai tinggi (25%)”. Ini terlalu kasar untuk strategi produk, dan karena itu aturan rentang bisnis 4–8 dibuat.
- Di rentang 4–8, **k = 5** punya silhouette tertinggi untuk **kedua** model (K-Means 0,311; GMM 0,305), dan stabilitasnya sangat tinggi (ARI bootstrap 0,996 & 0,965). Bukti pendukung: siku inertia di k = 5, BIC GMM mendatar setelah k = 5, dan pada k ≥ 7 muncul segmen kecil (≈3%) serta stabilitas menurun (K-Means 0,92; GMM 0,83).

**Pemilihan model (Bagian 9)**
- **K-Means menang di 6 dari 6 kriteria**: silhouette (0,312 vs 0,306), CH, DB, ARI bootstrap (0,996 vs 0,982), Jaccard cluster terlemah (0,994 vs 0,952), dan split-half ARI (0,993 vs 0,981).
- Perbedaan stabilitas paling jelas: GMM **kadang** menemukan solusi yang berbeda (split-half ARI minimum 0,736 dan bootstrap P5 0,818), sedangkan K-Means konsisten (minimum 0,989). Wilcoxon split-half p = 0,012, jadi perbedaannya signifikan.
- Hanya 1,5% nasabah K-Means yang silhouette-nya negatif (GMM 3,1%).
- Uji permutasi: silhouette asli 0,31 vs data nol 0,08 (z = 104 untuk K-Means), **p = 0,01**. Ini nilai p terkecil yang mungkin dengan 99 permutasi: tidak ada satu pun data nol yang mendekati. Strukturnya nyata.

> **Catatan jujur:** validasi eksternal di Bagian 12 menunjukkan GMM sebenarnya **lebih mirip** persona asli (ARI 0,997 vs 0,944). Metrik internal mengukur kekompakan & stabilitas, bukan “kebenaran”. Di data nyata kebenaran tidak diketahui, jadi keputusan tetap mengikuti aturan yang ditetapkan di awal. Selisih praktisnya kecil: 97,5% nasabah K-Means tetap mendapat persona yang benar.

## 10. Model Final & Penamaan Segmen Otomatis

Nomor cluster bersifat acak (cluster “0” tidak berarti apa-apa), jadi nama persona **tidak di-hardcode per nomor**. `src/personas.py` berisi aturan berbasis profil yang dievaluasi berurutan, dan setiap persona mengambil cluster dengan skor tertinggi yang belum bernama:

| Urutan | Persona | Aturan (pada median fitur per cluster) |
|---|---|---|
| 1 | Nasabah Prioritas (Affluent) | `investment_balance` tertinggi |
| 2 | Pelaku Usaha / UMKM | `monthly_txn_count` tertinggi |
| 3 | Nasabah Bergantung Kredit | rasio `loan_outstanding / monthly_salary` tertinggi |
| 4 | Profesional Muda Digital | `digital_txn_ratio − umur/100` tertinggi |
| 5 | Nasabah Massal Penabung | `monthly_salary` terendah |

Dengan cara ini, jika model dilatih ulang dan nomor cluster berubah, nama persona tetap konsisten.

> **🔄 Alur data: Model Final & Penamaan Persona**
>
> - **Input:** `candidates[best_name]`, `dev`.
> - **Proses:**
>   1. Label dev → median 11 fitur numerik per cluster.
>   2. `name_segments` menerapkan aturan persona berurutan (greedy) → `SEGMENTS` (nama, kode, deskripsi, rekomendasi).
> - **Output:** `final_model`, `K` = 5, `dev_labels`, `SEGMENTS`, `seg_name`, `seg_code`. Porsi: Massal Penabung 35%, Muda Digital 22%, Bergantung Kredit 19%, UMKM 14%, Affluent 10%.
> - **Berikutnya:** Bangun profil lengkap tiap segmen.

In [ ]:
final_model = candidates[best_name]
K = BEST_K[best_name]
dev_labels = final_model.predict(dev[FEATURES])
dev_profile_median = dev[NUMERIC_COLS].assign(segment=dev_labels).groupby("segment").median()
SEGMENTS = name_segments(dev_profile_median)
seg_name = {c: s["name"] for c, s in SEGMENTS.items()}
seg_code = {c: s["code"] for c, s in SEGMENTS.items()}

pd.DataFrame({
    "kode": seg_code, "nama": seg_name, "aturan": {c: s["rule"] for c, s in SEGMENTS.items()},
    "jumlah": pd.Series(dev_labels).value_counts().sort_index(), "porsi": pd.Series(dev_labels).value_counts(normalize=True).sort_index().round(3),
})

## 11. Profil Segmen

### 11.1 Profil numerik

> **🔄 Alur data: Profil Numerik & Indeks Relatif**
>
> - **Input:** `dev`, `dev_labels`, `seg_name`.
> - **Proses:**
>   1. `add_profile_bins` menambah `age_band` & `monthly_salary_band` (binning profil). Hasilnya `dev_seg`.
>   2. Median 11 fitur per segmen + porsi nasabah + median rasio pinjaman/gaji → `profile`.
>   3. Indeks relatif = median segmen / median seluruh nasabah, digambar sebagai heatmap (skala log2).
> - **Output:** `dev_seg`, `profile`, heatmap indeks. Affluent: investasi 598 jt (indeks ratusan kali median bank), saldo 259 jt. UMKM: 137 transaksi/bulan senilai 100 jt. Bergantung Kredit: pinjaman 99 jt dengan gaji 7,6 jt. Massal Penabung: semua indikator nilai di bawah median.
> - **Berikutnya:** Lihat posisi segmen di peta PCA.

In [ ]:
dev_seg = add_profile_bins(dev).assign(segment=dev_labels, segment_name=lambda d: d["segment"].map(seg_name))
profile = dev_seg.groupby("segment_name")[NUMERIC_COLS].median()
profile.insert(0, "porsi_nasabah", dev_seg["segment_name"].value_counts(normalize=True))
profile["loan_to_salary"] = dev_seg.groupby("segment_name").apply(lambda g: (g["loan_outstanding"] / g["monthly_salary"]).median(), include_groups=False)
profile = profile.sort_values("monthly_salary")
display(profile.round(2))

# indeks relatif: median segmen / median seluruh nasabah (1 = sama dengan rata-rata bank)
overall = dev[NUMERIC_COLS].median().replace(0, np.nan)
rel = (profile[NUMERIC_COLS] / overall).round(2)
fig, ax = plt.subplots(figsize=(14, 4.5))
sns.heatmap(np.log2(rel.astype(float).clip(lower=0.05)), annot=rel, fmt="", cmap="RdBu_r", center=0, ax=ax,
            cbar_kws={"label": "log2(indeks)"})
ax.set_title("Indeks median segmen vs median seluruh nasabah (angka = kali lipat; merah = di atas rata-rata)")
plt.tight_layout()
plt.show()

> **🔄 Alur data: Peta Segmen (PCA 2D)**
>
> - **Input:** `pca_dev`, data dev ter-transformasi, `dev_labels`.
> - **Proses:**
>   1. Setiap nasabah diproyeksikan ke 2 komponen utama dan diwarnai per segmen; tanda X = median segmen.
> - **Output:** Peta PCA. PC1 ≈ sumbu “nilai nasabah”: Massal Penabung paling kiri (median PC1 −2,3), Bergantung Kredit di tengah (−0,2), UMKM di kanan (2,6), Affluent paling kanan-atas (4,0; PC2 2,1). Muda Digital terpisah di **bawah** (PC2 −1,8), karena PC2 menangkap dimensi usia muda & digital.
> - **Berikutnya:** Profil dengan binning.

In [ ]:
centers_2d = pca_dev.transform(preprocessor.transform(dev[FEATURES]))
fig, ax = plt.subplots(figsize=(10, 7))
for c in range(K):
    m = dev_labels == c
    ax.scatter(centers_2d[m, 0], centers_2d[m, 1], s=4, alpha=0.35, label=f"{seg_name[c]} ({m.mean():.0%})")
    ax.scatter(*np.median(centers_2d[m], axis=0), s=250, marker="X", edgecolors="k")
ax.set_xlabel(f"PC1 ({pca_dev.explained_variance_ratio_[0]:.0%})")
ax.set_ylabel(f"PC2 ({pca_dev.explained_variance_ratio_[1]:.0%})")
ax.set_title("Segmen di ruang PCA 2D (X = median segmen)")
ax.legend(markerscale=4)
plt.tight_layout()
plt.show()

### 11.2 Profil berdasarkan binning & demografi

Di sini binning dipakai untuk **membaca** segmen: kelompok umur dan rentang gaji (batas UMR) jauh lebih mudah dikomunikasikan ke tim bisnis daripada median angka.

> **🔄 Alur data: Profil Binning (Umur, Gaji, Kanal)**
>
> - **Input:** `dev_seg`.
> - **Proses:**
>   1. `pd.crosstab(normalize="index")`: komposisi kelompok umur, rentang gaji, dan kanal di setiap segmen (stacked bar 100%).
> - **Output:** Grafik komposisi + tabel rentang gaji. 61% Massal Penabung bergaji < 5 jt; 76% Affluent bergaji 20–50 jt (20% > 50 jt); Muda Digital 68% berumur 26–35 dan 84% memakai kanal mobile; Massal Penabung paling banyak memakai ATM (41%) & cabang (29%).
> - **Berikutnya:** Profil demografi yang tidak dipakai model.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 5))
for ax, col in zip(axes, ["age_band", "monthly_salary_band", "preferred_channel"]):
    ct = pd.crosstab(dev_seg["segment_name"], dev_seg[col], normalize="index")
    ct.plot.barh(stacked=True, ax=ax, colormap="viridis", width=0.8)
    ax.set_title(f"Komposisi {col} per segmen")
    ax.legend(fontsize=7, bbox_to_anchor=(1, 1))
    ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()
display((pd.crosstab(dev_seg["segment_name"], dev_seg["monthly_salary_band"], normalize="index") * 100).round(1))

> **🔄 Alur data: Profil Demografi**
>
> - **Input:** `dev_seg`: `occupation`, `employee_status`, `city_tier`.
> - **Proses:**
>   1. Komposisi (%) setiap kolom demografi per segmen.
> - **Output:** 3 tabel. UMKM 77% `entrepreneur`; Bergantung Kredit 40% PNS & 34% karyawan swasta (78% karyawan tetap); Massal Penabung = buruh pabrik 35%, retail 30%, driver 19%; Affluent 74% di kota tier-1.
> - **Berikutnya:** Uji statistik perbedaan antar segmen.

In [ ]:
for col in PROFILE_CAT_COLS:
    ct = (pd.crosstab(dev_seg["segment_name"], dev_seg[col], normalize="index") * 100).round(1)
    print(f"Komposisi {col} per segmen (%)")
    display(ct)

### 11.3 Uji Statistik Perbedaan Antar Segmen

- **Kruskal-Wallis H** (non-parametrik, cocok untuk data miring): H0 = median fitur sama di semua segmen. Besar efek **ε²**: < 0,01 sangat kecil, 0,01–0,08 kecil, 0,08–0,26 sedang, ≥ 0,26 besar.
- **Chi-square + Cramér's V** untuk kolom kategorikal (termasuk hasil binning dan kolom demografi yang **tidak** dipakai model).
- p-value dikoreksi **Holm**.

> ⚠️ Fitur yang dipakai model **pasti** berbeda antar segmen karena memang itu tujuan clustering, jadi p-value-nya *by construction* kecil. Yang lebih informatif adalah: (1) **besar efek**, yaitu fitur mana yang paling membedakan segmen, dan (2) uji pada **kolom demografi yang tidak dipakai model**. Jika kolom demografi juga berbeda, segmen tersebut bermakna di luar fitur pembentuknya.

> **🔄 Alur data: Uji Beda Antar Segmen**
>
> - **Input:** `dev_seg`.
> - **Proses:**
>   1. 11 fitur numerik: Kruskal-Wallis H + ε².
>   2. Kanal, 3 kolom demografi, 2 kolom binning: chi-square + Cramér's V.
>   3. p-value dikoreksi Holm; kolom diberi tanda apakah dipakai model.
> - **Output:** `seg_tests` (17 baris). Semua signifikan (p_holm < 0,001). Fitur model: ε² 0,31–0,82 (besar). Kolom yang **tidak** dipakai model: `occupation` V = **0,63**, `employee_status` 0,40, `city_tier` 0,29.
> - **Berikutnya:** Validasi eksternal dengan label persona asli.

In [ ]:
rows = []
for col in NUMERIC_COLS:
    h, p, eps2 = kruskal_epsilon(dev_seg[col], dev_seg["segment"])
    rows.append({"fitur": col, "dipakai_model": "ya", "uji": "Kruskal-Wallis", "statistik": h, "p_value": p, "besar_efek": eps2,
                 "kategori_efek": pd.cut([eps2], [0, 0.01, 0.08, 0.26, 1], labels=["sangat kecil", "kecil", "sedang", "besar"], include_lowest=True)[0]})
for col in CAT_COLS + PROFILE_CAT_COLS + ["age_band", "monthly_salary_band"]:
    chi2, p, v = cramers_v(dev_seg[col], dev_seg["segment"])
    used = "ya" if col in CAT_COLS else ("turunan" if col.endswith("_band") else "tidak")
    rows.append({"fitur": col, "dipakai_model": used, "uji": "Chi-square", "statistik": chi2, "p_value": p, "besar_efek": v,
                 "kategori_efek": pd.cut([v], [0, 0.1, 0.3, 0.5, 1], labels=["lemah", "sedang", "kuat", "sangat kuat"], include_lowest=True)[0]})
seg_tests = pd.DataFrame(rows).set_index("fitur")
seg_tests["p_holm"] = holm_correction(seg_tests["p_value"])
seg_tests["signifikan"] = np.where(seg_tests["p_holm"] < 0.05, "ya", "tidak")
seg_tests.sort_values(["uji", "besar_efek"], ascending=[False, False]).round(4)

### Interpretasi Profil Segmen

**5 segmen yang ditemukan (K-Means, k = 5):**

| Segmen | Porsi | Ciri utama (median) |
|---|---|---|
| **Nasabah Massal Penabung** | 35% | gaji 4,5 jt (sekitar UMR), saldo 5,5 jt, investasi ≈ 0, 2 produk, 15 transaksi/bulan, rasio digital terendah (0,47), tidak memakai kartu kredit |
| **Profesional Muda Digital** | 22% | usia 28, rasio digital 0,94, 64 transaksi/bulan, gaji 10,9 jt, nasabah baru (2 tahun) |
| **Nasabah Bergantung Kredit** | 19% | pinjaman 99 jt dengan gaji 7,6 jt (±13× gaji), saldo hanya 3,8 jt, kartu kredit 5,3 jt/bulan |
| **Pelaku Usaha / UMKM** | 14% | 137 transaksi/bulan senilai 100 jt, saldo 90 jt, pinjaman 94 jt |
| **Nasabah Prioritas (Affluent)** | 10% | gaji 38 jt, saldo 259 jt, investasi 598 jt, 6 produk, usia 53, nasabah 12 tahun |

**Uji statistik**
- Semua 11 fitur model berbeda sangat signifikan dengan efek **besar** (ε² 0,31–0,82). Ini memang tujuan clustering. Pembeda terkuat: **frekuensi & nominal transaksi** (ε² 0,82 & 0,81), lalu gaji dan saldo.
- Yang lebih penting: **kolom demografi yang TIDAK dipakai model juga berbeda tegas**. `occupation` (Cramér's V = **0,63**, sangat kuat), `employee_status` (0,40), dan `city_tier` (0,29). Segmen yang dibentuk dari perilaku transaksi ternyata juga mencerminkan siapa nasabahnya: UMKM didominasi wiraswasta, Bergantung Kredit didominasi PNS & karyawan swasta, Affluent di kota tier-1. Ini **validasi silang** bahwa segmen bermakna di luar fitur pembentuknya.
- **Binning memperjelas komunikasi**: mayoritas Massal Penabung bergaji < 5 jt, sedangkan mayoritas Affluent di atas 20 jt. Profesional Muda Digital terkonsentrasi di kelompok umur 26–35.

## 12. Validasi Eksternal (Khusus Data Sintetis)

Data ini dibangkitkan dari 5 persona yang diketahui (`_true_persona`), jadi bisa dicek apakah clustering **menemukan kembali** persona tersebut. Kolom ini **tidak** dipakai sama sekali saat memilih k atau model.

- **ARI (Adjusted Rand Index)**: 0 = sama dengan acak, 1 = identik.
- **NMI (Normalized Mutual Information)**: 0–1, informasi bersama antara dua pelabelan.

> Di data nyata validasi ini **tidak mungkin** dilakukan. Bagian ini hanya membuktikan bahwa prosedur evaluasi internal di Bagian 8–9 memang mengarah ke jawaban yang benar.

> **🔄 Alur data: Validasi Eksternal**
>
> - **Input:** `cand_labels` (kedua kandidat), `dev[_true_persona]`, `dev_seg`.
> - **Proses:**
>   1. ARI & NMI setiap kandidat vs persona asli.
>   2. Crosstab segmen hasil model × persona asli, dan porsi nasabah yang nama segmennya cocok dengan persona asli.
> - **Output:** `ext`. ARI vs persona asli: K-Means **0,944**, GMM **0,997**. Crosstab: kesalahan K-Means terutama 84 nasabah Bergantung Kredit & 49 UMKM yang masuk Muda Digital. **97,5%** nasabah mendapat nama persona yang benar.
> - **Berikutnya:** Mulai validasi out-of-time.

In [ ]:
ext = pd.DataFrame({
    MODEL_LABELS[n]: {"k": BEST_K[n], "ARI": adjusted_rand_score(dev[TRUE_LABEL_COL], lab), "NMI": normalized_mutual_info_score(dev[TRUE_LABEL_COL], lab)}
    for n, lab in cand_labels.items()}).T
display(ext.round(4))
ct = pd.crosstab(dev_seg["segment_name"], dev[TRUE_LABEL_COL])
display(ct)
name_match = {seg_name[c]: seg_code[c] for c in SEGMENTS}
purity = sum(ct.loc[n, name_match[n]] for n in ct.index if name_match[n] in ct.columns) / len(dev)
print(f"Porsi nasabah yang nama segmennya cocok dengan persona asli: {purity:.1%}")

## 13. Validasi Out-of-Time (Backtesting Segmentasi)

Model yang dilatih di **Jan–Sep 2025** dipakai untuk memberi label nasabah **Okt–Des 2025**. Ada empat hal yang diuji:

| # | Uji | Pertanyaan |
|---|---|---|
| 13.1 | Distribusi segmen dev vs OOT: **chi-square**, **PSI**, CI Wilson | Apakah komposisi segmen berubah? |
| 13.2 | Kualitas cluster di OOT: silhouette, jarak ke pusat segmen | Apakah segmen tetap kompak untuk data baru? |
| 13.3 | **Refit di OOT** + pencocokan Hungarian: ARI & drift centroid | Jika dilatih ulang dengan data baru, apakah segmen yang sama muncul lagi? |
| 13.4 | **CSI** per fitur | Fitur mana yang bergeser? |

> **🔄 Alur data: OOT: Komposisi Segmen**
>
> - **Input:** `final_model` (dilatih di dev), `oot`.
> - **Proses:**
>   1. Nasabah OOT diberi label oleh model dev (`predict`, tanpa fit ulang).
>   2. Porsi segmen dev vs OOT + 95% CI Wilson untuk porsi OOT.
>   3. Chi-square (H0: komposisi sama) dan PSI distribusi segmen.
> - **Output:** `oot_labels`, `dist_table`. Muda Digital 22,0% → **26,6%**, Massal Penabung 35,4% → **31,5%**; segmen lain stabil. χ² = 33,6, p < 0,001 (signifikan), tetapi **PSI = 0,015 (stabil)**.
> - **Berikutnya:** Cek apakah segmen tetap kompak di OOT.

In [ ]:
oot_labels = final_model.predict(oot[FEATURES])
dist_table = pd.DataFrame({
    "dev": pd.Series(dev_labels).value_counts(normalize=True).sort_index(),
    "OOT": pd.Series(oot_labels).value_counts(normalize=True).sort_index(),
}).fillna(0)
dist_table.index = dist_table.index.map(seg_name)
dist_table["selisih_pp"] = (dist_table["OOT"] - dist_table["dev"]) * 100
ci = [stats.binomtest(int(round(p * len(oot))), len(oot)).proportion_ci(method="wilson") for p in dist_table["OOT"]]
dist_table["OOT_ci_lower"], dist_table["OOT_ci_upper"] = [c.low for c in ci], [c.high for c in ci]
dist_table["dev_di_dalam_CI_OOT"] = np.where((dist_table["dev"] >= dist_table["OOT_ci_lower"]) & (dist_table["dev"] <= dist_table["OOT_ci_upper"]), "ya", "tidak")

chi2_seg, p_seg, _, _ = stats.chi2_contingency(pd.crosstab(np.r_[np.zeros(len(dev_labels)), np.ones(len(oot_labels))], np.r_[dev_labels, oot_labels]))
segment_psi = psi_categorical(dev_labels, oot_labels)
display(dist_table.round(4))
print(f"Chi-square komposisi segmen dev vs OOT: χ² = {chi2_seg:.1f}, p = {p_seg:.4g}")
print(f"PSI distribusi segmen: {segment_psi:.4f} ({stability_label(segment_psi)})")

ax = dist_table[["dev", "OOT"]].plot.barh(figsize=(9, 4.5), color=["#4C72B0", "#C44E52"])
ax.set_title("Komposisi segmen: dev (Jan–Sep) vs OOT (Okt–Des)")
plt.tight_layout()
plt.show()

> **🔄 Alur data: OOT: Kualitas Cluster**
>
> - **Input:** `X_oot` (OOT ditransformasi dengan preprocessor dev), `X_dev_final`, label dev & OOT.
> - **Proses:**
>   1. Silhouette dev vs OOT.
>   2. Jarak setiap nasabah ke pusat segmennya: median per segmen dev vs OOT, dan uji Mann-Whitney untuk keseluruhan.
> - **Output:** `X_oot`, `compact`. Silhouette dev 0,312 → OOT **0,318** (+0,006). Jarak ke pusat segmen: rasio OOT/dev 0,99–1,00; Mann-Whitney p = 0,23.
> - **Berikutnya:** Cek apakah model baru di OOT menemukan segmen yang sama.

In [ ]:
X_oot = final_model.named_steps["preprocess"].transform(oot[FEATURES])
X_dev_final = final_model.named_steps["preprocess"].transform(dev[FEATURES])
sil_dev = internal_metrics(X_dev_final, dev_labels)["silhouette"]
sil_oot = internal_metrics(X_oot, oot_labels)["silhouette"]

model_obj = final_model.named_steps["model"]
centers = model_obj.cluster_centers_ if best_name == "kmeans" else model_obj.means_
d_dev = np.linalg.norm(X_dev_final - centers[dev_labels], axis=1)
d_oot = np.linalg.norm(X_oot - centers[oot_labels], axis=1)
mw = stats.mannwhitneyu(d_dev, d_oot)
compact = pd.DataFrame({
    "median_jarak_dev": pd.Series(d_dev).groupby(dev_labels).median(),
    "median_jarak_OOT": pd.Series(d_oot).groupby(oot_labels).median(),
})
compact.index = compact.index.map(seg_name)
compact["rasio_OOT/dev"] = compact["median_jarak_OOT"] / compact["median_jarak_dev"]
display(compact.round(3))
print(f"Silhouette dev = {sil_dev:.4f} | OOT = {sil_oot:.4f} | selisih = {sil_oot - sil_dev:+.4f}")
print(f"Jarak ke pusat segmen: median dev {np.median(d_dev):.3f} vs OOT {np.median(d_oot):.3f}, Mann-Whitney p = {mw.pvalue:.4g}")

> **🔄 Alur data: OOT: Refit & Pencocokan Hungarian**
>
> - **Input:** `X_oot`, model final.
> - **Proses:**
>   1. Model dengan konfigurasi sama dilatih **ulang** hanya di OOT.
>   2. Pusat segmen refit dicocokkan ke pusat segmen dev dengan algoritma Hungarian (total jarak minimum), lalu label refit dipetakan ke nomor segmen dev.
>   3. ARI & % kesepakatan label model dev vs refit di data OOT; drift centroid per segmen (satuan std).
> - **Output:** `oot_refit`, `match`, `drift`. ARI dev-model vs refit-OOT = **0,981**; **99,2%** label sama. Drift centroid 0,21–0,31 std total (±0,05–0,08 std per fitur).
> - **Berikutnya:** Cari fitur penyebab pergeseran.

In [ ]:
oot_refit = clone(model_obj).fit(X_oot)
oot_refit_centers = oot_refit.cluster_centers_ if best_name == "kmeans" else oot_refit.means_
match = match_clusters(centers, oot_refit_centers)
oot_refit_labels = pd.Series(oot_refit.predict(X_oot)).map(match["mapping"]).to_numpy()
refit_ari = adjusted_rand_score(oot_labels, oot_refit_labels)
agreement = (oot_labels == oot_refit_labels).mean()

drift = pd.DataFrame({"segmen": [seg_name[c] for c in range(K)],
                      "drift_centroid_(std)": [np.linalg.norm(centers[c] - oot_refit_centers[[k for k, v in match["mapping"].items() if v == c][0]]) for c in range(K)]})
drift["drift_per_fitur_rms"] = drift["drift_centroid_(std)"] / np.sqrt(centers.shape[1])
display(drift.set_index("segmen").round(4))
print(f"ARI label model dev vs model refit OOT (di data OOT): {refit_ari:.4f}")
print(f"Nasabah OOT yang mendapat segmen sama dari kedua model: {agreement:.1%}")

> **🔄 Alur data: OOT: CSI Fitur & Pergeseran Digital**
>
> - **Input:** `dev`, `oot`.
> - **Proses:**
>   1. CSI tiap fitur model & kolom profil (numerik: 10 kuantil dev; kategorikal: per kategori).
>   2. Rata-rata `digital_txn_ratio` per segmen dev vs OOT.
> - **Output:** `csi_table`, `digital_shift`. Hanya `digital_txn_ratio` CSI **0,148 (perlu perhatian)**; fitur lain ≤ 0,012. Rasio digital naik +0,04 s.d. +0,06 di **semua** segmen.
> - **Berikutnya:** Terjemahkan segmen ke nilai bisnis.

In [ ]:
csi_rows = []
for col in FEATURES + PROFILE_CAT_COLS:
    value = psi_categorical(dev[col], oot[col]) if col in CAT_COLS + PROFILE_CAT_COLS else psi_numeric(dev[col], oot[col])
    csi_rows.append({"fitur": col, "CSI": value, "status": stability_label(value),
                     "dipakai_model": "ya" if col in FEATURES else "tidak"})
csi_table = pd.DataFrame(csi_rows).set_index("fitur").sort_values("CSI", ascending=False)

digital_shift = pd.DataFrame({
    "digital_ratio_dev": dev.groupby(dev_labels)["digital_txn_ratio"].mean(),
    "digital_ratio_OOT": oot.groupby(oot_labels)["digital_txn_ratio"].mean(),
})
digital_shift.index = digital_shift.index.map(seg_name)
digital_shift["selisih"] = digital_shift["digital_ratio_OOT"] - digital_shift["digital_ratio_dev"]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(csi_table.index, csi_table["CSI"], color=np.where(csi_table["dipakai_model"] == "ya", "#4C72B0", "#BBBBBB"))
ax.axvline(0.1, color="#E1B12C", ls="--", label="0,1")
ax.axvline(0.25, color="#C44E52", ls="--", label="0,25")
ax.invert_yaxis()
ax.set_title("CSI dev → OOT (biru = dipakai model, abu = profil)")
ax.legend()
plt.tight_layout()
plt.show()
display(csi_table.round(4))
display(digital_shift.round(3))

### Interpretasi Validasi Out-of-Time

**13.1 Komposisi berubah, signifikan tetapi belum material**
- Porsi **Profesional Muda Digital naik 22,0% → 26,6%** (+4,7 poin) dan **Massal Penabung turun 35,4% → 31,5%** (−3,9 poin). Untuk kedua segmen ini, porsi dev berada di luar 95% CI Wilson porsi OOT. Tiga segmen lain stabil.
- Chi-square: χ² = 33,6, **p < 0,001**, jadi perubahan komposisi **signifikan secara statistik**. Namun PSI distribusi segmen hanya **0,015 (stabil)**. Dengan ±12.000 nasabah, perubahan beberapa poin persen pun signifikan. Secara bisnis ini **tren yang perlu dicatat**, yaitu pergeseran nasabah ke digital, bukan tanda model rusak.

**13.2 Segmen tetap kompak**
- Silhouette OOT **0,318** vs dev 0,312 (bahkan naik +0,006).
- Jarak median nasabah ke pusat segmennya praktis sama (rasio OOT/dev 0,99–1,00 di semua segmen; Mann-Whitney p = 0,23).

**13.3 Struktur bisa direproduksi dari data baru**
- Model yang dilatih **ulang hanya dengan data OOT** menemukan segmen yang sama: ARI **0,981**, dan **99,2%** nasabah OOT mendapat segmen yang sama dari model dev maupun model refit.
- Drift centroid per segmen kecil: 0,21–0,31 std total, atau ±0,05–0,08 std per fitur.

**13.4 Sumber pergeseran: adopsi digital**
- Satu-satunya fitur dengan CSI > 0,1 adalah **`digital_txn_ratio` (0,148, perlu perhatian)**. Fitur lain < 0,02.
- Kenaikan rasio digital terjadi di **semua** segmen (+0,04 s.d. +0,06), jadi ini tren pasar, bukan perubahan satu segmen. Implikasinya: dalam 1–2 kuartal ke depan, batas antar segmen pada dimensi digital akan bergeser, sehingga model perlu **dilatih ulang berkala** (mis. setiap 6 bulan) atau dipantau dengan CSI.

**Kesimpulan OOT:** segmentasi **berlaku** di periode berikutnya. Strukturnya stabil dan kualitasnya tidak turun. Perubahan yang terdeteksi (porsi digital naik) adalah sinyal bisnis yang berguna dan berhasil ditangkap oleh model.

## 14. Rekomendasi Bisnis & Ringkasan Evaluasi

### 14.1 Nilai bisnis tiap segmen

> **🔄 Alur data: Nilai Bisnis per Segmen**
>
> - **Input:** `dev_seg`, `SEGMENTS`.
> - **Proses:**
>   1. Per segmen: jumlah nasabah, total dana (saldo), total investasi, total kredit, dan total transaksi, dinyatakan sebagai % dari total bank.
>   2. Rekomendasi produk dari `SEGMENTS`.
> - **Output:** `biz`. Affluent: 10% nasabah, **55% dana, 92% investasi**. UMKM: 14% nasabah, **60% transaksi**, 33% kredit. Bergantung Kredit: 19% nasabah, **44% kredit**, 1,5% dana. Massal Penabung: 35% nasabah, 4% dana.
> - **Berikutnya:** Rangkum semua bukti statistik.

In [ ]:
biz = dev_seg.groupby("segment_name").agg(
    jumlah_nasabah=("customer_id", "size"),
    total_dana=("avg_balance", "sum"),
    total_investasi=("investment_balance", "sum"),
    total_kredit=("loan_outstanding", "sum"),
    total_transaksi=("monthly_txn_amount", "sum"),
)
for col in ["jumlah_nasabah", "total_dana", "total_investasi", "total_kredit", "total_transaksi"]:
    biz[f"%_{col}"] = biz[col] / biz[col].sum() * 100
code_by_name = {v: seg_code[k] for k, v in seg_name.items()}
biz["rekomendasi"] = [" • ".join(SEGMENTS[[c for c in SEGMENTS if seg_name[c] == n][0]]["recommendations"]) for n in biz.index]
pd.set_option("display.max_colwidth", 200)
display(biz[[c for c in biz.columns if c.startswith("%_")]].round(1).sort_values("%_total_investasi", ascending=False))
display(biz[["rekomendasi"]])

### 14.2 Ringkasan Evaluasi Statistik

> **🔄 Alur data: Ringkasan Evaluasi Statistik**
>
> - **Input:** Hasil Bagian 6–13.
> - **Proses:**
>   1. 16 pengujian dibandingkan dengan patokan → status ✅/⚠️/❌ → `evaluation_summary`.
>   2. Angka kunci dikemas ke `EVALUATION_FOR_METADATA` untuk `model.json`.
> - **Output:** `evaluation_summary` (16 baris): **15 ✅, 1 ⚠️** (CSI `digital_txn_ratio`), 0 ❌. `EVALUATION_FOR_METADATA` siap disimpan.
> - **Berikutnya:** Ekspor model ke JSON.

In [ ]:
def verdict(ok: bool, warn: bool = False) -> str:
    return "✅" if ok else ("⚠️" if warn else "❌")


b, sh, pr = boot[best_name], split_half[best_name], perm[best_name]
jac_min = b["jaccard_per_cluster"].mean().min()
max_model_csi = csi_table.loc[csi_table["dipakai_model"] == "ya", "CSI"].max()
max_model_csi_feat = csi_table.loc[csi_table["dipakai_model"] == "ya", "CSI"].idxmax()
demo_v = seg_tests.loc[PROFILE_CAT_COLS, "besar_efek"]
rows = [
    ("Kecenderungan cluster", "Hopkins statistic", f"{hopkins_real:.3f} (data nol {hopkins_null.mean():.3f})", "> 0,7", verdict(hopkins_real > 0.7, hopkins_real > 0.6)),
    ("Kecenderungan cluster", "Uji permutasi silhouette", f"{pr['silhouette_real']:.3f} vs nol {pr['null_mean']:.3f} (z = {pr['z']:.0f})", f"p = {pr['p_value']:.3f} < 0,05", verdict(pr["p_value"] < 0.05)),
    ("Pemilihan k", f"k terpilih {MODEL_LABELS[best_name]}", f"{K}", f"rentang bisnis {BUSINESS_K}, stabil, silhouette maks", verdict(True)),
    ("Kualitas internal", "Silhouette (seluruh dev)", f"{cand_internal.loc[MODEL_LABELS[best_name], 'silhouette']:.3f}", "> 0,25 (struktur wajar)", verdict(cand_internal.loc[MODEL_LABELS[best_name], 'silhouette'] > 0.25, True)),
    ("Kualitas internal", "Nasabah silhouette < 0", f"{sil_neg[best_name]:.1%}", "< 10%", verdict(sil_neg[best_name] < 0.10, sil_neg[best_name] < 0.2)),
    ("Kualitas internal", "Segmen terkecil", f"{cand_internal.loc[MODEL_LABELS[best_name], 'segmen_terkecil']:.1%}", "≥ 3% (actionable)", verdict(cand_internal.loc[MODEL_LABELS[best_name], 'segmen_terkecil'] >= 0.03)),
    ("Stabilitas (≈ CV)", f"Bootstrap ARI ({N_BOOT_FINAL}×)", f"{b['ari_mean']:.3f} ± {b['ari_std']:.3f}", "≥ 0,75", verdict(b["ari_mean"] >= 0.75, b["ari_mean"] >= 0.6)),
    ("Stabilitas (≈ CV)", "Jaccard cluster terlemah", f"{jac_min:.3f}", "≥ 0,75 semua cluster", verdict(jac_min >= 0.75, jac_min >= 0.6)),
    ("Stabilitas (≈ CV)", f"Split-half ARI ({N_SPLIT_HALF}×)", f"{sh.mean():.3f} (min {sh.min():.3f})", "≥ 0,75", verdict(sh.mean() >= 0.75, sh.mean() >= 0.6)),
    ("Pemilihan model", f"{MODEL_LABELS[best_name]} vs {MODEL_LABELS[other_name]}", f"menang {wins.get(MODEL_LABELS[best_name], 0)} dari {len(comparison)} kriteria", "mayoritas", verdict(wins.get(MODEL_LABELS[best_name], 0) > len(comparison) / 2, True)),
    ("Makna segmen", "Demografi (tidak dipakai model) berbeda antar segmen", f"Cramér's V {demo_v.min():.2f}–{demo_v.max():.2f}", "V ≥ 0,1 & p_holm < 0,05", verdict((demo_v >= 0.1).all() and (seg_tests.loc[PROFILE_CAT_COLS, 'p_holm'] < 0.05).all(), True)),
    ("Validasi eksternal*", "ARI vs persona asli", f"{ext.loc[MODEL_LABELS[best_name], 'ARI']:.3f}", "> 0,8 (*hanya data sintetis)", verdict(ext.loc[MODEL_LABELS[best_name], 'ARI'] > 0.8, True)),
    ("Out-of-time", "PSI distribusi segmen", f"{segment_psi:.4f}", "< 0,1", verdict(segment_psi < 0.1, segment_psi < 0.25)),
    ("Out-of-time", "Perubahan silhouette dev → OOT", f"{sil_oot - sil_dev:+.4f}", "|Δ| < 0,03", verdict(abs(sil_oot - sil_dev) < 0.03, abs(sil_oot - sil_dev) < 0.05)),
    ("Out-of-time", "ARI model dev vs refit OOT", f"{refit_ari:.3f} ({agreement:.1%} sama)", "≥ 0,75", verdict(refit_ari >= 0.75, refit_ari >= 0.6)),
    ("Out-of-time", "CSI fitur model maksimum", f"{max_model_csi:.4f} ({max_model_csi_feat})", "< 0,1", verdict(max_model_csi < 0.1, max_model_csi < 0.25)),
]
evaluation_summary = pd.DataFrame(rows, columns=["aspek", "pengujian", "nilai", "patokan / p-value", "status"])
display(evaluation_summary.set_index(["aspek", "pengujian"]))
print("Rekap status:", evaluation_summary["status"].value_counts().to_dict())

EVALUATION_FOR_METADATA = {
    "hopkins": float(hopkins_real),
    "permutation_test": {"silhouette": float(pr["silhouette_real"]), "null_mean": float(pr["null_mean"]), "p_value": float(pr["p_value"])},
    "k_scan_rule": f"k in {list(BUSINESS_K)}, bootstrap ARI >= {STABILITY_MIN}, max silhouette",
    "silhouette_dev": float(sil_dev), "silhouette_oot": float(sil_oot),
    "bootstrap_ari": {"mean": float(b["ari_mean"]), "std": float(b["ari_std"])},
    "jaccard_per_segment": {seg_code[c]: float(v) for c, v in b["jaccard_per_cluster"].mean().items()},
    "split_half_ari": {"mean": float(sh.mean()), "min": float(sh.min())},
    "model_comparison": {MODEL_LABELS[n]: {crit: float(vals[n]) for crit, (vals, _) in criteria.items()} for n in MODEL_FACTORIES},
    "oot": {"segment_psi": float(segment_psi), "chi2_p": float(p_seg), "refit_ari": float(refit_ari),
            "max_feature_csi": {"feature": max_model_csi_feat, "value": float(max_model_csi)}},
    "external_ari_synthetic_only": float(ext.loc[MODEL_LABELS[best_name], "ARI"]),
}

### Kesimpulan Akhir

**Hasil: 15 ✅ · 1 ⚠️ · 0 ❌**

**1. Segmentasi layak dipakai**, didukung bukti dari 5 sisi:

| Sisi | Bukti utama |
|---|---|
| **Struktur nyata** | Hopkins 0,79 (data nol 0,66); uji permutasi p = 0,01 (z = 104) |
| **Jumlah segmen** | k = 5 dipilih dengan aturan tertulis; didukung siku inertia, BIC teregularisasi, dan stabilitas |
| **Stabilitas (≈ cross-validation)** | ARI bootstrap 0,996; Jaccard cluster terlemah 0,994; split-half ARI 0,993 |
| **Makna bisnis** | Demografi yang tidak dipakai model ikut berbeda tegas (pekerjaan V = 0,63); 5 persona jelas & actionable |
| **Out-of-time** | Silhouette OOT 0,318 (≥ dev); refit OOT menghasilkan segmen yang sama (ARI 0,98); PSI segmen 0,015 |

**2. Satu catatan ⚠️:** `digital_txn_ratio` bergeser (CSI 0,148) karena adopsi digital naik di semua segmen. Ini **belum** merusak segmentasi, tetapi menjadi alasan untuk retrain berkala.

**3. Nilai bisnis yang terlihat jelas**
- **Affluent**: 10% nasabah memegang **55% dana dan 92% investasi**. Prioritas retensi #1.
- **UMKM**: 14% nasabah menghasilkan **60% nominal transaksi** dan 33% kredit. Sumber fee-based income & kredit produktif.
- **Bergantung Kredit**: 19% nasabah memegang **44% kredit**, tetapi hanya 1,5% dana. Fokus pada **manajemen risiko**, bukan cross-sell kredit.
- **Profesional Muda Digital**: segmen yang **tumbuh paling cepat** (22% → 27%). Investasi di produk mobile & life-stage (KPR pertama, investasi mikro).
- **Massal Penabung**: segmen terbesar (35%) dengan dana kecil. Biaya layanan harus rendah, dan nasabah perlu didorong ke kanal digital.

**4. Rekomendasi operasional**
1. Pakai **`segment_code`** (bukan nomor cluster) di CRM & campaign, karena kode tetap konsisten walaupun model dilatih ulang.
2. **Monitoring bulanan**: PSI distribusi segmen & CSI fitur (alarm 0,1/0,25), silhouette pada data baru, dan porsi tiap segmen.
3. **Retrain setiap 6 bulan**, atau lebih cepat jika CSI > 0,25 atau ARI refit < 0,75. Setelah retrain, cek ulang pemetaan persona dan perubahan porsi.
4. **Validasi dengan data riil**: data proyek ini sintetis. Di data nyata biasanya ada lebih banyak noise dan segmen yang tumpang tindih, sehingga silhouette akan lebih rendah. Prosedur evaluasi di notebook ini tetap berlaku.
5. Jika tim bisnis butuh **tingkat keyakinan per nasabah** (mis. untuk nasabah di perbatasan dua segmen), GMM bisa dipertimbangkan karena memberi probabilitas dan secara eksternal lebih mirip persona asli. Konsekuensinya, stabilitas GMM sedikit lebih rendah.

## 15. Simpan Model dalam Format JSON

`models/model.json` berisi:
- `preprocessing`: nilai imputasi, flag log1p, mean & std scaler, kategori one-hot, dan bobotnya
- `model`: centroid (K-Means) atau bobot, mean, dan precision Cholesky (GMM)
- `metadata.segments`: nama, deskripsi, rekomendasi, porsi, dan profil median setiap segmen, sehingga API bisa langsung mengembalikan persona
- `metadata.evaluation`: ringkasan bukti statistik dari Bagian 14

> **🔄 Alur data: Ekspor Model JSON**
>
> - **Input:** `final_model`, `SEGMENTS`, profil, label dev/OOT, `EVALUATION_FOR_METADATA`.
> - **Proses:**
>   1. Metadata per segmen: persona, porsi dev & OOT, median profil.
>   2. `export_pipeline` membaca parameter preprocessing & model yang sudah dipelajari → `save_json`.
> - **Output:** `model_json`, file **`models/model.json` (±11 KB)**, tipe model `kmeans` (5 centroid × 15 dimensi).
> - **Berikutnya:** Buktikan JSON identik dengan scikit-learn.

In [ ]:
now = datetime.now(timezone.utc)
seg_meta = {}
for c, s in SEGMENTS.items():
    seg_meta[str(c)] = {**s, "share_dev": float((dev_labels == c).mean()), "share_oot": float((oot_labels == c).mean()),
                        "profile_median": {k: float(v) for k, v in dev_profile_median.loc[c].items()}}
metadata = {
    "model_name": best_name,
    "model_version": now.strftime("%Y%m%d%H%M%S"),
    "trained_at": now.isoformat(timespec="seconds"),
    "training_period": f"{dev[DATE_COL].min()} – {dev[DATE_COL].max()}",
    "n_training_rows": int(len(dev)),
    "n_clusters": int(K),
    "features": FEATURES,
    "numeric_features": NUMERIC_COLS,
    "categorical_features": {col: sorted(dev[col].dropna().unique().tolist()) for col in CAT_COLS},
    "segments": seg_meta,
    "evaluation": EVALUATION_FOR_METADATA,
    "library_versions": {"python": platform.python_version(), "scikit-learn": sklearn.__version__,
                         "pandas": pd.__version__, "numpy": np.__version__},
}
model_json = export_pipeline(final_model, metadata)
model_path = MODEL_DIR / "model.json"
save_json(model_json, model_path)
print(f"Model JSON -> {model_path} ({model_path.stat().st_size / 1024:.1f} KB)")
print("Bagian:", list(model_json), "| tipe model:", model_json["model"]["type"])

## 16. Validasi Model JSON & Contoh Request API

> **🔄 Alur data: Validasi Model JSON**
>
> - **Input:** `models/model.json` (dibaca ulang dari disk), `dev`, `oot`, `final_model`.
> - **Proses:**
>   1. `JsonSegmentModel` (hanya numpy/pandas) memprediksi dev & OOT, lalu dibandingkan dengan pipeline scikit-learn.
>   2. Kasus khusus: 50 nasabah dengan gaji kosong & kanal tak dikenal (`whatsapp`).
> - **Output:** Label JSON vs scikit-learn sama **100%** di dev, OOT, dan kasus khusus; selisih transformasi 0.
> - **Berikutnya:** Buat contoh request API.

In [ ]:
json_model = JsonSegmentModel.load(model_path)
checks = {}
for label, data in [("dev", dev), ("OOT", oot)]:
    checks[f"label sama ({label})"] = (json_model.predict(data[FEATURES]) == final_model.predict(data[FEATURES])).mean()
checks["selisih transformasi maks"] = np.abs(json_model.transform(oot[FEATURES]) - X_oot).max()
if best_name == "gmm":
    checks["selisih probabilitas maks"] = np.abs(json_model.predict_proba(oot[FEATURES]) - final_model.predict_proba(oot[FEATURES])).max()

edge = oot[FEATURES].head(50).copy()
edge["monthly_salary"] = np.nan
edge["preferred_channel"] = "whatsapp"   # kanal yang belum pernah dilihat
checks["label sama (kasus khusus)"] = (json_model.predict(edge) == final_model.predict(edge)).mean()
display(pd.Series(checks).to_frame("hasil"))
assert checks["label sama (dev)"] == 1 and checks["label sama (OOT)"] == 1 and checks["label sama (kasus khusus)"] == 1
print("✅ Model JSON identik dengan pipeline scikit-learn")

> **🔄 Alur data: Contoh Request API**
>
> - **Input:** `oot`, `json_model`, `oot_labels`.
> - **Proses:**
>   1. Untuk setiap segmen dipilih nasabah OOT yang paling dekat ke pusat segmennya (contoh paling “khas”).
>   2. Dikonversi ke JSON (NaN → null) dan disimpan sebagai file contoh request.
> - **Output:** `examples` (5 nasabah “khas”, satu per segmen), file `sample_request.json` (contoh UMKM) dan `sample_batch_request.json` (5 nasabah).
> - **Berikutnya:** File dipakai untuk mencoba API dan unit test.

In [ ]:
def to_payload(row: pd.Series) -> dict:
    return {k: (None if pd.isna(v) else (v.item() if hasattr(v, "item") else v)) for k, v in row.items()}


# satu contoh per segmen: nasabah OOT yang paling dekat ke pusat segmennya
dist_oot = json_model.distances(oot[FEATURES])
examples = []
for c in range(K):
    idx = np.where(oot_labels == c)[0]
    best = idx[np.argmin(dist_oot[idx, c])]
    examples.append({"customer_id": oot.loc[best, ID_COL], **to_payload(oot.loc[best, FEATURES])})

(PROJECT_DIR / "sample_request.json").write_text(json.dumps(examples[0], indent=2))
(PROJECT_DIR / "sample_batch_request.json").write_text(json.dumps({"instances": examples}, indent=2))
pd.DataFrame(examples).assign(segmen=[seg_name[c] for c in json_model.predict(pd.DataFrame(examples))])

## 17. Alur Data di API

```
POST /segment  (12 field perilaku nasabah + customer_id opsional)
   │ 1. Pydantic (app/schemas.py): field lengkap, tipe, rentang (mis. digital_txn_ratio 0–1),
   │    kanal valid; monthly_salary boleh null → kalau gagal: HTTP 422
   ▼
DataFrame (n baris)
   │ 2. JsonSegmentModel.transform: isi NaN (median dev) → log1p → (x − mean)/std
   │                                 + one-hot kanal × 0,5         → matriks n × 15
   ▼
   │ 3. K-Means: segmen = centroid terdekat; GMM: probabilitas posterior tiap komponen
   ▼
   │ 4. metadata.segments[segmen] → nama persona, deskripsi, rekomendasi produk
   ▼
Response: segment_id, segment_code, segment_name, confidence, scores per segmen,
          distance_to_center, description, recommendations
```

**Siklus berikutnya:** data baru → jalankan ulang notebook → `model.json` baru → restart/rebuild API. Karena persona dinamai berdasarkan aturan profil, kode segmen (`affluent`, `sme_transactor`, …) tetap konsisten walaupun nomor cluster berubah. Sistem lain (CRM, campaign) sebaiknya memakai **`segment_code`**, bukan `segment_id`.